# tts-workbench: training notebook (Kaggle & Colab)

Train any of the text-to-speech models of the `tts-workbench` repo on a **free Kaggle or Colab GPU**:

| `MODEL` | family | text -> | needs a vocoder |
|---|---|---|---|
| `tacotron2` | autoregressive, location-sensitive attention | mel | yes |
| `transformer_tts` | autoregressive, transformer encoder-decoder | mel | yes |
| `fastspeech2` | non-autoregressive, built-in aligner, duration/pitch/energy | mel | yes |
| `vits` | end-to-end VAE + flow + GAN | waveform | no |
| `hifigan` | neural vocoder (train it on the same data, then chain it with a mel model) | waveform | - |

This notebook is **self-contained**: all the code is copied in from the repo with the same
defaults, and its checkpoints work with the repo's `tts-synthesize` and `tts-eval`.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings -> Accelerator -> GPU T4 x2* (or P100) | *Runtime -> Change runtime type -> T4 GPU* |
| Data | your dataset as input, or `DOWNLOAD_DATASET` | Google Drive, upload, or `DOWNLOAD_DATASET` |
| Session limit | 12 h per session, 30 h GPU / week | ~12 h, can disconnect when idle |
| Checkpoints kept in | `/kaggle/working` (saved with "Save Version") | Google Drive if `USE_GOOGLE_DRIVE = True` |

**Just trying it?** Keep `SMOKE_TEST = True`: the notebook writes the toy language dataset and
runs every cell in a few minutes. For real speech set `DOWNLOAD_DATASET = "ljspeech"` (2.6 GB, 24 h
of one English speaker) and the 22.05 kHz audio settings, or `"fsdd"` / `"yesno"` for small 8 kHz tests.

**Sections:** 1 Install - 2 Config - 3 Text & audio - 4 Building blocks & alignment - 5 Models -
6 Vocoders & losses - 7 Data - 8 Training code - 9 Train - 10 Results (listen, MCD/F0/WER/UTMOS) -
11 Download the checkpoint

## 1. Install

Kaggle and Colab already have PyTorch, NumPy, SciPy and matplotlib. This cell adds `soundfile`
(audio I/O) and `transformers` (Whisper, for the intelligibility WER in section 10).

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "soundfile>=0.12", "transformers>=4.40"])

import torch

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Config

### 2.1 Config classes (`src/tts/config.py`)

Configuration: small dataclasses, overridable from the command line.

Every field of every dataclass becomes a `--flag-name` option (`AudioConfig.sample_rate`
-> `--sample-rate`). Defaults are deliberately small: these models are built to be
understood and trained on a laptop CPU or a free Kaggle GPU, not to beat benchmarks.
Scale `hidden`, `layers` and `batch_size` up for real data.

In [ ]:
from __future__ import annotations

import argparse
import dataclasses
import json
from dataclasses import dataclass, field, fields
from pathlib import Path
from typing import Any

MODELS = ("tacotron2", "transformer_tts", "fastspeech2", "vits", "hifigan")
VOCODERS = ("griffinlim", "hifigan")


@dataclass
class AudioConfig:
    """How waveforms become (log-)mel-spectrograms and back."""

    sample_rate: int = 8000  # 8 kHz default matches the toy / FSDD / YESNO datasets; use 22050 for LJSpeech
    n_fft: int = 512
    hop_length: int = 128  # 16 ms at 8 kHz
    win_length: int = 512
    n_mels: int = 64
    fmin: float = 0.0
    fmax: float = 4000.0  # <= sample_rate / 2
    log_floor: float = 1e-5  # log(max(mel, floor))
    f0_min: float = 60.0
    f0_max: float = 400.0


@dataclass
class TextConfig:
    """Text front-end."""

    tokenizer: str = "char"  # "char" | "phoneme" (needs g2p-en)
    lowercase: bool = True


@dataclass
class ModelConfig:
    """Architecture sizes shared by all models (each model reads what it needs)."""

    model: str = "tacotron2"  # one of MODELS
    hidden: int = 128  # encoder/decoder width (Tacotron2 paper: 512)
    layers: int = 2  # FFT / transformer blocks per stack
    heads: int = 2
    ffn_kernel: int = 3
    dropout: float = 0.1
    prenet_dim: int = 64
    attention_dim: int = 64
    attention_filters: int = 16
    attention_kernel: int = 15
    reduction_factor: int = 2  # Tacotron2 / Transformer-TTS: mel frames per decoder step
    postnet_layers: int = 3
    postnet_channels: int = 128
    n_speakers: int = 1  # > 1 enables a speaker embedding (FiLM-style add) in every model
    speaker_dim: int = 32
    max_decoder_steps: int = 600  # inference safety stop for autoregressive models
    # VITS / HiFi-GAN
    latent_dim: int = 32
    flow_blocks: int = 2
    upsample_rates: tuple[int, ...] = (8, 4, 4)  # product must equal AudioConfig.hop_length
    upsample_initial: int = 64
    resblock_kernels: tuple[int, ...] = (3, 7)
    resblock_dilations: tuple[int, ...] = (1, 3)
    segment_frames: int = 16  # frames of random audio segment fed to the waveform decoder in GAN training


@dataclass
class TrainConfig:
    """Optimisation."""

    data_dir: str = "data/synthetic"
    output_dir: str = "checkpoints/run"
    epochs: int = 20
    max_steps: int = 0  # 0 = epochs decide
    batch_size: int = 16
    lr: float = 1e-3
    warmup_steps: int = 200
    weight_decay: float = 1e-6
    grad_clip: float = 1.0
    amp: bool = False  # bf16/fp16 autocast on GPU
    num_workers: int = 0
    seed: int = 0
    log_every: int = 10
    eval_every: int = 1  # epochs
    time_budget_hours: float = 0.0  # 0 = no limit
    resume: bool = False
    duration_source: str = "aligner"  # fastspeech2: "aligner" | "teacher" (durations cached by tts-eval --export-durations)
    guided_attention: float = 1.0  # weight of the diagonal guided-attention loss (AR models), 0 = off
    max_frames: int = 600  # clips longer than this (in mel frames) are skipped
    device: str = "auto"


@dataclass
class Config:
    audio: AudioConfig = field(default_factory=AudioConfig)
    text: TextConfig = field(default_factory=TextConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)

    def to_dict(self) -> dict[str, Any]:
        return dataclasses.asdict(self)

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> Config:
        cfg = cls()
        for section in fields(cls):
            sub = getattr(cfg, section.name)
            for f in fields(sub):
                if f.name in d.get(section.name, {}):
                    value = d[section.name][f.name]
                    if isinstance(value, list):
                        value = tuple(value)
                    setattr(sub, f.name, value)
        return cfg

    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(self.to_dict(), indent=2))

    @classmethod
    def load(cls, path: str | Path) -> Config:
        return cls.from_dict(json.loads(Path(path).read_text()))


def _parse_value(text: str, default: Any) -> Any:
    if isinstance(default, bool):
        return text.lower() in ("1", "true", "yes", "on")
    if isinstance(default, tuple):
        return tuple(int(x) for x in text.split(","))
    if isinstance(default, int):
        return int(text)
    if isinstance(default, float):
        return float(text)
    return text


def add_config_args(parser: argparse.ArgumentParser, cfg: Config | None = None) -> None:
    """One `--flag` per dataclass field; the help shows the default."""
    cfg = cfg or Config()
    for section in fields(Config):
        group = parser.add_argument_group(section.name)
        sub = getattr(cfg, section.name)
        for f in fields(sub):
            default = getattr(sub, f.name)
            group.add_argument(f"--{f.name.replace('_', '-')}", dest=f"{section.name}.{f.name}", default=None,
                               metavar=type(default).__name__.upper(), help=f"default: {default}")


def config_from_args(args: argparse.Namespace, base: Config | None = None) -> Config:
    """Apply the given `--flags` on top of `base` (or the defaults)."""
    cfg = base or Config()
    for key, value in vars(args).items():
        if "." not in key or value is None:
            continue
        section, name = key.split(".", 1)
        sub = getattr(cfg, section)
        setattr(sub, name, _parse_value(value, getattr(sub, name)))
    return cfg

### 2.2 Settings for this run

**Where is your data?** A folder in the LJSpeech layout (`metadata.csv` + `wavs/`, optional
`speakers.csv`), see the repo's `docs/data.md`.

- **Kaggle:** add the dataset as input; the cell below finds it.
- **Colab:** put it on Google Drive (`USE_GOOGLE_DRIVE = True`) and set `DATA_DIR`.
- **Either:** set `DOWNLOAD_DATASET` to `"ljspeech"`, `"fsdd"` or `"yesno"`.

`AUDIO` must match the data: 8 kHz for the small sets, 22.05 kHz / 80 mels for LJSpeech.

In [ ]:
import os
from pathlib import Path

SMOKE_TEST = True            # True = toy language data, small model, a few minutes
MODEL = "tacotron2"          # "tacotron2" | "transformer_tts" | "fastspeech2" | "vits" | "hifigan"
VOCODER = "griffinlim"       # "griffinlim" | "hifigan" (needs HIFIGAN_CHECKPOINT) - ignored by vits
HIFIGAN_CHECKPOINT = None    # path of a hifigan best.pt trained with this notebook (MODEL = "hifigan")
DOWNLOAD_DATASET = None      # None | "ljspeech" (2.6 GB) | "fsdd" (20 MB) | "yesno" (4.5 MB)
USE_GOOGLE_DRIVE = False     # Colab only: data + checkpoints on Drive (survive disconnects)
DATA_DIR = None              # None = find it automatically (Kaggle input, /content/data, ./data)
RESUME = True                # continue from last.pt in OUTPUT_DIR if it exists
TIME_BUDGET_HOURS = 11.0     # stop cleanly (after saving) before the 12 h session limit
EPOCHS = 200
BATCH_SIZE = 32
LR = 1e-3
AUDIO = dict(sample_rate=22050, n_fft=1024, hop_length=256, win_length=1024, n_mels=80, fmax=8000.0)
UPSAMPLE_RATES = (8, 8, 4)   # hifigan / vits: product must equal hop_length (8*8*4 = 256)
if SMOKE_TEST or DOWNLOAD_DATASET in ("fsdd", "yesno"):
    AUDIO = dict(sample_rate=8000, n_fft=512, hop_length=128, win_length=512, n_mels=64, fmax=4000.0)
    UPSAMPLE_RATES = (8, 4, 4)

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
if ON_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path(f"/content/drive/MyDrive/tts-workbench/{MODEL}")
elif ON_KAGGLE:
    OUTPUT_DIR = Path(f"/kaggle/working/checkpoints/{MODEL}")
elif ON_COLAB:
    OUTPUT_DIR = Path(f"/content/checkpoints/{MODEL}")
else:
    OUTPUT_DIR = Path(f"checkpoints/notebook-{MODEL}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_data_dir():
    roots = [Path("/kaggle/input"), Path("/content/data"), Path("/content/drive/MyDrive"), Path("data")]
    for root in roots:
        if not root.exists():
            continue
        for path in [root, *sorted(p for p in root.rglob("*") if p.is_dir())][:500]:
            if (path / "metadata.csv").exists() and (path / "wavs").is_dir():
                return path
    return None


print("output:", OUTPUT_DIR)

## 3. Text & audio

### 3.1 Text front-end (`src/tts/text.py`)

Text front-end: normalisation and tokenisation.

A TTS model never sees raw strings. The text is first *normalised* (lower-cased, small
numbers spelled out, punctuation reduced to a few symbols) and then *tokenised* into
integer ids: one id per character (`char`) or per phoneme (`phoneme`, via `g2p-en`).
Id 0 is padding, id 1 is the end-of-sentence symbol `~` that every sentence ends with
(it gives autoregressive models a clean signal to stop).

In [ ]:
from __future__ import annotations

import re

_ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven",
         "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
_TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]


def number_to_words(n: int) -> str:
    """0 <= n < 1_000_000 as English words (enough for dates and counts in captions)."""
    if n < 20:
        return _ONES[n]
    if n < 100:
        return _TENS[n // 10] + ("" if n % 10 == 0 else " " + _ONES[n % 10])
    if n < 1000:
        return _ONES[n // 100] + " hundred" + ("" if n % 100 == 0 else " " + number_to_words(n % 100))
    if n < 1_000_000:
        return number_to_words(n // 1000) + " thousand" + ("" if n % 1000 == 0 else " " + number_to_words(n % 1000))
    return str(n)


def normalize(text: str, lowercase: bool = True) -> str:
    """Spell out integers, keep letters, apostrophes, spaces and . , ? ! ; collapse spaces."""
    if lowercase:
        text = text.lower()
    text = re.sub(r"\d+", lambda m: number_to_words(int(m.group())) if len(m.group()) < 7 else m.group(), text)
    text = text.replace("-", " ").replace(";", ",").replace(":", ",")
    text = re.sub(r"[^a-z' .,?!]", " ", text) if lowercase else re.sub(r"[^A-Za-z' .,?!]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


PAD, EOS = "_", "~"
CHAR_SYMBOLS = [PAD, EOS, " ", "'", ".", ",", "?", "!"] + list("abcdefghijklmnopqrstuvwxyz")

# ARPAbet (CMUdict) phonemes, as produced by g2p-en, stress digits stripped.
ARPABET = ["AA", "AE", "AH", "AO", "AW", "AY", "B", "CH", "D", "DH", "EH", "ER", "EY", "F", "G", "HH", "IH", "IY",
           "JH", "K", "L", "M", "N", "NG", "OW", "OY", "P", "R", "S", "SH", "T", "TH", "UH", "UW", "V", "W", "Y",
           "Z", "ZH"]
PHONEME_SYMBOLS = [PAD, EOS, " ", ".", ",", "?", "!"] + ARPABET


class Tokenizer:
    """Text -> list of ids (and back). `kind` is "char" or "phoneme"."""

    def __init__(self, kind: str = "char", lowercase: bool = True):
        self.kind = kind
        self.lowercase = lowercase
        self.symbols = CHAR_SYMBOLS if kind == "char" else PHONEME_SYMBOLS
        self.index = {s: i for i, s in enumerate(self.symbols)}
        self._g2p = None

    @property
    def vocab_size(self) -> int:
        return len(self.symbols)

    @property
    def pad_id(self) -> int:
        return 0

    @property
    def eos_id(self) -> int:
        return 1

    def _phonemes(self, text: str) -> list[str]:
        if self._g2p is None:
            from g2p_en import G2p  # optional extra: pip install "tts-workbench[phonemes]"

            self._g2p = G2p()
        out = []
        for p in self._g2p(text):
            p = p.rstrip("012")
            if p in self.index:
                out.append(p)
        return out

    def encode(self, text: str) -> list[int]:
        text = normalize(text, self.lowercase)
        units = list(text) if self.kind == "char" else self._phonemes(text)
        ids = [self.index[u] for u in units if u in self.index]
        return ids + [self.eos_id]

    def decode(self, ids: list[int]) -> str:
        sep = "" if self.kind == "char" else " "
        return sep.join(self.symbols[i] for i in ids if i > 1)

### 3.2 Mel-spectrograms, Griffin-Lim, F0 (`src/tts/audio.py`)

Audio: loading, mel-spectrograms, Griffin-Lim, F0 and energy.

The *mel-spectrogram* is the intermediate representation of every acoustic model here:
`|STFT|` (linear magnitude, n_fft/2+1 bins) -> mel filterbank (n_mels bins on a
perceptual frequency scale) -> log. Shapes are `(batch, n_mels, frames)` with
`frames = samples // hop_length` (the signal is padded by (n_fft - hop) / 2 on each side).

Griffin-Lim turns a magnitude spectrogram back into a waveform by alternating between
"take the STFT phase of the current guess" and "keep the target magnitude"; it needs no
training, which makes it the baseline vocoder (and sounds metallic, which is why
neural vocoders exist).

F0 is estimated with a normalised-autocorrelation (YIN-like) method per frame; FastSpeech 2
predicts it and the metrics compare it between synthesis and recording.

In [ ]:
from __future__ import annotations

import math
from functools import lru_cache

import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly



def load_wav(path: str, sample_rate: int) -> np.ndarray:
    """Mono float32 waveform at `sample_rate` (resampled with a polyphase filter if needed)."""
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != sample_rate:
        g = math.gcd(sr, sample_rate)
        wav = resample_poly(wav, sample_rate // g, sr // g).astype(np.float32)
    peak = np.abs(wav).max()
    if peak > 1.0:
        wav = wav / peak
    return wav


def save_wav(path: str, wav: np.ndarray | torch.Tensor, sample_rate: int) -> None:
    if isinstance(wav, torch.Tensor):
        wav = wav.detach().cpu().numpy()
    wav = np.clip(wav, -1.0, 1.0).astype(np.float32)
    sf.write(path, wav, sample_rate)


def hz_to_mel(f: float) -> float:
    return 2595.0 * math.log10(1.0 + f / 700.0)


def mel_to_hz(m: float) -> float:
    return 700.0 * (10.0 ** (m / 2595.0) - 1.0)


@lru_cache(maxsize=8)
def mel_filterbank(sample_rate: int, n_fft: int, n_mels: int, fmin: float, fmax: float) -> torch.Tensor:
    """Triangular (Slaney-style) filters, shape (n_mels, n_fft // 2 + 1)."""
    n_bins = n_fft // 2 + 1
    freqs = np.linspace(0, sample_rate / 2, n_bins)
    mels = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_mels + 2)
    hz = np.array([mel_to_hz(m) for m in mels])
    fb = np.zeros((n_mels, n_bins), dtype=np.float32)
    for i in range(n_mels):
        lo, mid, hi = hz[i], hz[i + 1], hz[i + 2]
        up = (freqs - lo) / max(mid - lo, 1e-6)
        down = (hi - freqs) / max(hi - mid, 1e-6)
        fb[i] = np.clip(np.minimum(up, down), 0, None)
    enorm = 2.0 / (hz[2:] - hz[:-2])  # area normalisation so every filter has the same energy
    fb *= enorm[:, None]
    return torch.from_numpy(fb)


class MelSpectrogram:
    """Waveform (batch, samples) -> log-mel (batch, n_mels, frames); also linear magnitude for VITS."""

    def __init__(self, cfg: AudioConfig):
        self.cfg = cfg
        self.window = torch.hann_window(cfg.win_length)
        self.fb = mel_filterbank(cfg.sample_rate, cfg.n_fft, cfg.n_mels, cfg.fmin, cfg.fmax)

    def linear(self, wav: torch.Tensor) -> torch.Tensor:
        """Magnitude spectrogram (batch, n_fft // 2 + 1, frames)."""
        c = self.cfg
        pad = (c.n_fft - c.hop_length) // 2
        wav = torch.nn.functional.pad(wav.unsqueeze(1), (pad, pad), mode="reflect").squeeze(1)
        spec = torch.stft(wav, c.n_fft, c.hop_length, c.win_length, self.window.to(wav.device), center=False,
                          return_complex=True)
        return spec.abs()

    def mel_from_linear(self, mag: torch.Tensor) -> torch.Tensor:
        mel = torch.matmul(self.fb.to(mag.device), mag)
        return torch.log(torch.clamp(mel, min=self.cfg.log_floor))

    def __call__(self, wav: torch.Tensor) -> torch.Tensor:
        return self.mel_from_linear(self.linear(wav))

    def frames(self, n_samples: int) -> int:
        return n_samples // self.cfg.hop_length


def griffin_lim(log_mel: torch.Tensor, cfg: AudioConfig, n_iter: int = 32, power: float = 1.2) -> torch.Tensor:
    """Log-mel (n_mels, frames) -> waveform (samples). Mel is inverted with the pseudo-inverse
    of the filterbank, then the phase is recovered by Griffin-Lim iterations."""
    fb = mel_filterbank(cfg.sample_rate, cfg.n_fft, cfg.n_mels, cfg.fmin, cfg.fmax).to(log_mel.device)
    mag = torch.clamp(torch.linalg.pinv(fb) @ torch.exp(log_mel), min=0.0) ** power
    window = torch.hann_window(cfg.win_length, device=log_mel.device)
    angles = torch.exp(2j * math.pi * torch.rand(mag.shape, device=log_mel.device))
    spec = mag * angles
    for _ in range(n_iter):
        wav = torch.istft(spec, cfg.n_fft, cfg.hop_length, cfg.win_length, window)
        rebuilt = torch.stft(wav, cfg.n_fft, cfg.hop_length, cfg.win_length, window, return_complex=True)
        spec = mag * rebuilt / torch.clamp(rebuilt.abs(), min=1e-8)
    wav = torch.istft(spec, cfg.n_fft, cfg.hop_length, cfg.win_length, window)
    return wav / torch.clamp(wav.abs().max(), min=1e-5) * 0.95


def estimate_f0(wav: np.ndarray, cfg: AudioConfig, threshold: float = 0.3) -> np.ndarray:
    """Per-frame fundamental frequency in Hz (0 = unvoiced), aligned with the mel frames.

    For each frame the normalised autocorrelation is computed over lags between
    sample_rate/f0_max and sample_rate/f0_min; the best lag above `threshold` gives the pitch.
    """
    n_frames = len(wav) // cfg.hop_length
    win = cfg.win_length
    half = win // 2
    padded = np.pad(wav, (half, half + win), mode="constant")
    lag_min = max(2, int(cfg.sample_rate / cfg.f0_max))
    lag_max = min(win // 2, int(cfg.sample_rate / cfg.f0_min))
    f0 = np.zeros(n_frames, dtype=np.float32)
    for i in range(n_frames):
        start = i * cfg.hop_length
        frame = padded[start:start + win]
        frame = frame - frame.mean()
        energy = np.dot(frame, frame)
        if energy < 1e-6:
            continue
        # normalised autocorrelation for lags [lag_min, lag_max]
        spec = np.fft.rfft(frame, 2 * win)
        ac = np.fft.irfft(spec * np.conj(spec))[:win]
        ac = ac / (energy + 1e-9)
        seg = ac[lag_min:lag_max + 1]
        if seg.size == 0:
            continue
        k = int(np.argmax(seg))
        if seg[k] > threshold:
            lag = lag_min + k
            # parabolic interpolation for sub-sample precision
            if 0 < k < seg.size - 1:
                a, b, c = seg[k - 1], seg[k], seg[k + 1]
                denom = a - 2 * b + c
                if abs(denom) > 1e-9:
                    lag = lag + 0.5 * (a - c) / denom
            f0[i] = cfg.sample_rate / lag
    return f0


def frame_energy(mel: torch.Tensor) -> torch.Tensor:
    """Per-frame energy from a log-mel (n_mels, frames): log of the summed linear mel."""
    return torch.log(torch.clamp(torch.exp(mel).sum(dim=0), min=1e-5))

## 4. Building blocks & alignment

### 4.1 Shared layers (`src/tts/layers.py`)

Building blocks shared by the models.

- `PreNet`: two dropout-ed linear layers on the previous mel frame (Tacotron 2). The dropout
  stays on at inference: it is what stops the decoder from simply copying its last frame.
- `PostNet`: a stack of 1-D convolutions that predicts a residual to sharpen the coarse mel.
- `LocationSensitiveAttention`: Tacotron 2's attention. The score for each text position
  depends on the query, the encoder output *and the attention of the previous step*
  (convolved), which pushes the alignment to move forward monotonically.
- `FFTBlock`: the FastSpeech "feed-forward transformer" block: multi-head self-attention
  + a 1-D convolution feed-forward (convolutions see neighbouring phonemes, which linear
  layers cannot).
- `LengthRegulator`: repeats each phoneme's hidden state `duration` times (the non-
  autoregressive way to go from N phonemes to T frames).
- `VariancePredictor`: 2 conv layers + linear that predict duration / pitch / energy per phoneme.
- `SpeakerEmbedding`: a learned vector per speaker, added to the encoder output.

In [ ]:
from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn


def sequence_mask(lengths: torch.Tensor, max_len: int | None = None) -> torch.Tensor:
    """(batch, max_len) bool, True where t < length."""
    max_len = max_len or int(lengths.max())
    return torch.arange(max_len, device=lengths.device)[None, :] < lengths[:, None]


class ConvNorm(nn.Module):
    """Conv1d with 'same' padding, optional activation and dropout."""

    def __init__(self, cin: int, cout: int, kernel: int = 1, dilation: int = 1, act: nn.Module | None = None,
                 dropout: float = 0.0, norm: bool = True):
        super().__init__()
        self.conv = nn.Conv1d(cin, cout, kernel, padding=(kernel - 1) // 2 * dilation, dilation=dilation)
        self.norm = nn.BatchNorm1d(cout) if norm else nn.Identity()
        self.act = act or nn.Identity()
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.act(self.norm(self.conv(x))))


class PreNet(nn.Module):
    def __init__(self, cin: int, dim: int, dropout: float = 0.5):
        super().__init__()
        self.fc1, self.fc2, self.p = nn.Linear(cin, dim), nn.Linear(dim, dim), dropout

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # dropout is applied even in eval mode (Tacotron 2): it keeps the decoder from copying
        x = F.dropout(F.relu(self.fc1(x)), self.p, training=True)
        return F.dropout(F.relu(self.fc2(x)), self.p, training=True)


class PostNet(nn.Module):
    """n_mels -> channels -> ... -> n_mels residual, tanh except the last layer."""

    def __init__(self, n_mels: int, channels: int, layers: int, kernel: int = 5, dropout: float = 0.1):
        super().__init__()
        dims = [n_mels] + [channels] * (layers - 1) + [n_mels]
        self.convs = nn.ModuleList(
            ConvNorm(dims[i], dims[i + 1], kernel, act=nn.Tanh() if i < layers - 1 else None, dropout=dropout)
            for i in range(layers))

    def forward(self, mel: torch.Tensor) -> torch.Tensor:  # (batch, n_mels, frames)
        x = mel
        for conv in self.convs:
            x = conv(x)
        return x


class LocationSensitiveAttention(nn.Module):
    """e_{t,i} = v^T tanh(W q_t + V h_i + U f_{t,i}),   f_t = conv(alpha_{t-1} ++ sum_{<t} alpha)."""

    def __init__(self, query_dim: int, memory_dim: int, attn_dim: int, filters: int, kernel: int):
        super().__init__()
        self.query = nn.Linear(query_dim, attn_dim, bias=False)
        self.memory = nn.Linear(memory_dim, attn_dim, bias=False)
        self.location_conv = nn.Conv1d(2, filters, kernel, padding=(kernel - 1) // 2, bias=False)
        self.location = nn.Linear(filters, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, query: torch.Tensor, memory: torch.Tensor, processed_memory: torch.Tensor,
                prev_attn: torch.Tensor, cum_attn: torch.Tensor, mask: torch.Tensor):
        """query (B, Q); memory (B, N, M); prev/cum attn (B, N); mask (B, N) True = valid.
        Returns context (B, M) and alignment (B, N)."""
        loc = self.location_conv(torch.stack([prev_attn, cum_attn], dim=1)).transpose(1, 2)  # (B, N, filters)
        energies = self.v(torch.tanh(self.query(query)[:, None] + processed_memory + self.location(loc))).squeeze(-1)
        energies = energies.masked_fill(~mask, -1e4)
        alignment = torch.softmax(energies, dim=-1)
        context = torch.bmm(alignment.unsqueeze(1), memory).squeeze(1)
        return context, alignment


class PositionalEncoding(nn.Module):
    def __init__(self, dim: int, max_len: int = 4000):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(max_len)[:, None].float()
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)
        self.alpha = nn.Parameter(torch.ones(1))  # Transformer-TTS: learned scale of the encoding

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, T, dim)
        return x + self.alpha * self.pe[: x.size(1)]


class FFTBlock(nn.Module):
    """Self-attention + conv feed-forward, pre-norm residuals, key padding mask."""

    def __init__(self, dim: int, heads: int, kernel: int, dropout: float):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Conv1d(dim, dim * 4, kernel, padding=(kernel - 1) // 2), nn.ReLU(),
                                nn.Dropout(dropout), nn.Conv1d(dim * 4, dim, kernel, padding=(kernel - 1) // 2))
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:  # mask (B, T) True = valid
        h = self.ln1(x)
        h, _ = self.attn(h, h, h, key_padding_mask=~mask, need_weights=False)
        x = x + self.drop(h)
        h = self.ff(self.ln2(x).transpose(1, 2)).transpose(1, 2)
        x = x + self.drop(h)
        return x * mask[..., None]


class LengthRegulator(nn.Module):
    """(B, N, D) + integer durations (B, N) -> (B, T, D) with T = max(sum durations)."""

    def forward(self, x: torch.Tensor, durations: torch.Tensor):
        out, lengths = [], durations.sum(dim=1)
        max_len = int(lengths.max())
        for b in range(x.size(0)):
            expanded = torch.repeat_interleave(x[b], durations[b], dim=0)
            out.append(F.pad(expanded, (0, 0, 0, max_len - expanded.size(0))))
        return torch.stack(out), lengths


class VariancePredictor(nn.Module):
    def __init__(self, dim: int, filters: int = 128, kernel: int = 3, dropout: float = 0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(dim, filters, kernel, padding=(kernel - 1) // 2), nn.ReLU(), nn.GroupNorm(1, filters),
            nn.Dropout(dropout), nn.Conv1d(filters, filters, kernel, padding=(kernel - 1) // 2), nn.ReLU(),
            nn.GroupNorm(1, filters), nn.Dropout(dropout))
        self.out = nn.Linear(filters, 1)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:  # (B, N, D) -> (B, N)
        h = self.net(x.transpose(1, 2)).transpose(1, 2)
        return self.out(h).squeeze(-1) * mask


class SpeakerEmbedding(nn.Module):
    """Learned speaker vectors, projected and added to encoder states (no-op for n_speakers == 1)."""

    def __init__(self, n_speakers: int, speaker_dim: int, dim: int):
        super().__init__()
        self.enabled = n_speakers > 1
        if self.enabled:
            self.table = nn.Embedding(n_speakers, speaker_dim)
            self.proj = nn.Linear(speaker_dim, dim)

    def forward(self, x: torch.Tensor, speaker: torch.Tensor | None) -> torch.Tensor:  # x (B, N, dim)
        if not self.enabled or speaker is None:
            return x
        return x + self.proj(self.table(speaker))[:, None, :]


def guided_attention_loss(attn: torch.Tensor, text_lens: torch.Tensor, mel_lens: torch.Tensor,
                          sigma: float = 0.2) -> torch.Tensor:
    """Penalise attention far from the diagonal: W[n, t] = 1 - exp(-(n/N - t/T)^2 / 2 sigma^2).
    attn (B, T, N). Used in Transformer-TTS / Tacotron 2 training to make alignment learning fast."""
    _B, T, N = attn.shape
    n = torch.arange(N, device=attn.device)[None, None, :].float()
    t = torch.arange(T, device=attn.device)[None, :, None].float()
    w = 1.0 - torch.exp(-((n / text_lens[:, None, None] - t / mel_lens[:, None, None]) ** 2) / (2 * sigma**2))
    mask = sequence_mask(mel_lens, T)[:, :, None] & sequence_mask(text_lens, N)[:, None, :]
    return (attn * w * mask).sum() / mask.sum()

### 4.2 Monotonic alignment search and the aligner (`src/tts/align.py`)

Alignment between text and frames, without an external tool.

A non-autoregressive model (FastSpeech 2) needs to know, for each phoneme, how many
frames it lasts. Three ways to get these *durations* live here:

1. **Monotonic Alignment Search (MAS)**, from Glow-TTS. Given a log-likelihood matrix
   `logp[n, t]` (how well phoneme n explains frame t), dynamic programming finds the
   monotonic, non-skipping path N->T that maximises the total likelihood. It runs in
   O(N*T) with Python loops, which is fine at our sizes (`mas`).
2. **An aligner network** (as in "One TTS Alignment To Rule Them All"): a small text
   encoder and mel encoder give a soft alignment `softmax_n(-||mel_t - text_n||^2)`,
   trained with the *forward-sum* loss (the CTC forward algorithm over text positions:
   every monotonic path is summed, no blank). MAS over its log-probabilities yields hard
   durations. VITS uses the same MAS on the likelihood of its latent under the text prior.
3. **Teacher durations** from the attention matrix of a trained Tacotron 2
   (`durations_from_attention`): count how many frames each phoneme received the argmax.

In [ ]:
from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn



def mas(logp: np.ndarray, text_len: int, mel_len: int) -> np.ndarray:
    """Monotonic alignment search. logp (N, T) -> hard alignment (N, T) in {0, 1}.

    Each frame t is assigned to exactly one text position n, n never decreases and every
    text position gets at least one frame (needs T >= N).
    """
    neg = -1e9
    N, T = text_len, mel_len
    q = np.full((N, T), neg, dtype=np.float64)
    q[0, 0] = logp[0, 0]
    for t in range(1, T):
        lo, hi = max(0, N - T + t), min(N, t + 1)  # positions reachable at frame t
        for n in range(lo, hi):
            stay = q[n, t - 1]
            move = q[n - 1, t - 1] if n > 0 else neg
            q[n, t] = max(stay, move) + logp[n, t]
    path = np.zeros((N, T), dtype=np.int64)
    n = N - 1
    for t in range(T - 1, -1, -1):
        path[n, t] = 1
        if n > 0 and t > 0 and (q[n - 1, t - 1] > q[n, t - 1] or n == t):
            n -= 1
    return path


def mas_batch(logp: torch.Tensor, text_lens: torch.Tensor, mel_lens: torch.Tensor) -> torch.Tensor:
    """logp (B, N, T) -> (B, N, T) hard alignment (float, zeros outside each item's lengths)."""
    out = torch.zeros_like(logp)
    lp = logp.detach().cpu().numpy()
    for b in range(logp.size(0)):
        n, t = int(text_lens[b]), int(mel_lens[b])
        out[b, :n, :t] = torch.from_numpy(mas(lp[b, :n, :t], n, t)).to(logp.device, logp.dtype)
    return out


def forward_sum_loss(log_probs: torch.Tensor, text_lens: torch.Tensor, mel_lens: torch.Tensor) -> torch.Tensor:
    """log_probs (B, N, T) log-softmax over N per frame -> mean negative log of the sum of all monotonic
    paths. Implemented with CTC: text = [1..N] with no repeats and a blank that can never be used."""
    B, N, T = log_probs.shape
    # CTC wants (T, B, C) with class 0 = blank; shift text ids by one and give the blank -inf
    blank = torch.full((B, 1, T), -1e4, device=log_probs.device, dtype=log_probs.dtype)
    lp = torch.cat([blank, log_probs], dim=1).permute(2, 0, 1).log_softmax(-1)  # renormalise with blank
    targets = torch.arange(1, N + 1, device=log_probs.device)[None].expand(B, N)
    return F.ctc_loss(lp, targets, mel_lens, text_lens, blank=0, reduction="mean", zero_infinity=True)


class Aligner(nn.Module):
    """Text (B, N, D) and mel (B, n_mels, T) -> soft alignment (B, N, T) and log-probs."""

    def __init__(self, text_dim: int, n_mels: int, dim: int = 64):
        super().__init__()
        self.text = nn.Sequential(ConvNorm(text_dim, dim, 3, act=nn.ReLU(), norm=False), nn.Conv1d(dim, dim, 1))
        self.mel = nn.Sequential(ConvNorm(n_mels, dim, 3, act=nn.ReLU(), norm=False),
                                 ConvNorm(dim, dim, 3, act=nn.ReLU(), norm=False), nn.Conv1d(dim, dim, 1))

    def forward(self, text: torch.Tensor, mel: torch.Tensor, text_mask: torch.Tensor):
        h_text = self.text(text.transpose(1, 2))  # (B, D, N)
        h_mel = self.mel(mel)  # (B, D, T)
        dist = ((h_mel[:, :, None, :] - h_text[:, :, :, None]) ** 2).sum(1)  # (B, N, T)
        logits = -dist / h_text.size(1) ** 0.5
        logits = logits.masked_fill(~text_mask[:, :, None], -1e4)
        return logits.log_softmax(dim=1)  # over text positions


def durations_from_alignment(hard: torch.Tensor) -> torch.Tensor:
    """(B, N, T) hard alignment -> integer durations (B, N)."""
    return hard.sum(dim=2).long()


def durations_from_attention(attn: torch.Tensor, text_len: int, mel_len: int) -> torch.Tensor:
    """Teacher durations from an AR attention matrix (T, N): frames whose argmax is n, forced monotonic
    by MAS on the log-attention so every phoneme has >= 1 frame."""
    logp = torch.log(attn[:mel_len, :text_len].clamp_min(1e-8)).T.detach().cpu().numpy()
    return torch.from_numpy(mas(logp, text_len, mel_len).sum(axis=1))

## 5. Models

### 5.1 Tacotron 2 (`src/tts/tacotron2.py`)

Tacotron 2 (Shen et al. 2018): autoregressive attention-based text -> mel.

```
text ids -> Embedding -> 3 x Conv1d -> BiLSTM = encoder memory h (B, N, hidden)
frame 0 = zeros
for each decoder step t:
    p = PreNet(previous frame)                       # dropout always on
    attention LSTM: s_t = LSTM([p, context_{t-1}])
    alignment alpha_t = LocationSensitiveAttention(s_t, h, alpha_{t-1}, cumulative alpha)
    context_t = sum_i alpha_t[i] * h_i
    decoder LSTM: d_t = LSTM([s_t, context_t])
    mel_t (r frames) = Linear([d_t, context_t]);  stop_t = sigmoid(Linear([d_t, context_t]))
mel = mel + PostNet(mel)                            # residual refinement
```

Losses: MSE before and after the postnet, BCE on the stop token (positive at the last step),
and an optional guided-attention loss that pulls the alignment to the diagonal early on.
Training is *teacher-forced*: the previous frame comes from the recording, not from the
model; at inference the model feeds itself, which is why the prenet dropout matters.
`reduction_factor` r predicts r frames per step: fewer steps, easier attention.

In [ ]:
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn



class Tacotron2Encoder(nn.Module):
    def __init__(self, vocab: int, hidden: int, dropout: float):
        super().__init__()
        self.embed = nn.Embedding(vocab, hidden, padding_idx=0)
        self.convs = nn.Sequential(*[ConvNorm(hidden, hidden, 5, act=nn.ReLU(), dropout=dropout) for _ in range(3)])
        self.lstm = nn.LSTM(hidden, hidden // 2, batch_first=True, bidirectional=True)

    def forward(self, text: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        x = self.convs(self.embed(text).transpose(1, 2)).transpose(1, 2)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=text.size(1))
        return out


class Tacotron2(nn.Module):
    def __init__(self, cfg: Config, vocab_size: int):
        super().__init__()
        m, a = cfg.model, cfg.audio
        self.cfg, self.n_mels, self.r = cfg, a.n_mels, m.reduction_factor
        H = m.hidden
        self.encoder = Tacotron2Encoder(vocab_size, H, m.dropout)
        self.speaker = SpeakerEmbedding(m.n_speakers, m.speaker_dim, H)
        self.prenet = PreNet(a.n_mels, m.prenet_dim)
        self.attention_rnn = nn.LSTMCell(m.prenet_dim + H, H)
        self.attention = LocationSensitiveAttention(H, H, m.attention_dim, m.attention_filters, m.attention_kernel)
        self.decoder_rnn = nn.LSTMCell(2 * H, H)
        self.mel_out = nn.Linear(2 * H, a.n_mels * self.r)
        self.stop_out = nn.Linear(2 * H, self.r)
        self.postnet = PostNet(a.n_mels, m.postnet_channels, m.postnet_layers, dropout=m.dropout)
        self.dropout = m.dropout

    # -- one decoder step -------------------------------------------------
    def _init_state(self, memory: torch.Tensor):
        B, N, H = memory.shape
        z = memory.new_zeros
        return {"attn_h": z(B, H), "attn_c": z(B, H), "dec_h": z(B, H), "dec_c": z(B, H), "context": z(B, H),
                    "attn": z(B, N), "cum_attn": z(B, N)}

    def _step(self, prev_frame: torch.Tensor, state: dict, memory: torch.Tensor, processed: torch.Tensor,
              mask: torch.Tensor):
        p = self.prenet(prev_frame)
        state["attn_h"], state["attn_c"] = self.attention_rnn(torch.cat([p, state["context"]], -1),
                                                              (state["attn_h"], state["attn_c"]))
        query = F.dropout(state["attn_h"], self.dropout, self.training)
        context, attn = self.attention(query, memory, processed, state["attn"], state["cum_attn"], mask)
        state.update(context=context, attn=attn, cum_attn=state["cum_attn"] + attn)
        state["dec_h"], state["dec_c"] = self.decoder_rnn(torch.cat([query, context], -1),
                                                          (state["dec_h"], state["dec_c"]))
        out = torch.cat([F.dropout(state["dec_h"], self.dropout, self.training), context], -1)
        return self.mel_out(out), self.stop_out(out), attn

    def encode(self, text: torch.Tensor, text_lens: torch.Tensor, speaker: torch.Tensor | None):
        memory = self.speaker(self.encoder(text, text_lens), speaker)
        return memory, self.attention.memory(memory), sequence_mask(text_lens, text.size(1))

    # -- training (teacher forcing) --------------------------------------
    def forward(self, batch: dict) -> dict:
        mel, _mel_lens = batch["mel"], batch["mel_lens"]  # (B, n_mels, T)
        B, _, T = mel.shape
        pad = (-T) % self.r
        mel = F.pad(mel, (0, pad))
        T_pad = T + pad
        memory, processed, mask = self.encode(batch["text"], batch["text_lens"], batch.get("speaker"))
        state = self._init_state(memory)
        frames = mel.transpose(1, 2)  # (B, T_pad, n_mels)
        prev = frames.new_zeros(B, self.n_mels)
        mels, stops, attns = [], [], []
        for t in range(0, T_pad, self.r):
            m_out, s_out, attn = self._step(prev, state, memory, processed, mask)
            mels.append(m_out.view(B, self.r, self.n_mels))
            stops.append(s_out)
            attns.append(attn)
            prev = frames[:, t + self.r - 1]  # last frame of the group, from the recording
        mel_before = torch.cat(mels, 1).transpose(1, 2)[:, :, :T]
        stop_logits = torch.cat(stops, 1)[:, :T]
        mel_after = mel_before + self.postnet(mel_before)
        attn = torch.stack(attns, 1)  # (B, steps, N)
        return self.loss(mel_before, mel_after, stop_logits, attn, batch) | {
            "mel": mel_after, "attention": attn}

    def loss(self, before, after, stop_logits, attn, batch) -> dict:
        mel, mel_lens, text_lens = batch["mel"], batch["mel_lens"], batch["text_lens"]
        mask = sequence_mask(mel_lens, mel.size(2))[:, None, :]
        n = mask.sum() * mel.size(1)
        l_before = (F.mse_loss(before, mel, reduction="none") * mask).sum() / n
        l_after = (F.mse_loss(after, mel, reduction="none") * mask).sum() / n
        stop_target = (torch.arange(mel.size(2), device=mel.device)[None] >= (mel_lens - 1)[:, None]).float()
        l_stop = F.binary_cross_entropy_with_logits(stop_logits, stop_target, pos_weight=torch.tensor(5.0))
        total = l_before + l_after + l_stop
        out = {"mel_before": l_before, "mel_after": l_after, "stop": l_stop}
        if self.cfg.train.guided_attention > 0:
            steps = torch.div(mel_lens + self.r - 1, self.r, rounding_mode="floor")
            l_ga = guided_attention_loss(attn, text_lens, steps)
            total = total + self.cfg.train.guided_attention * l_ga
            out["guided_attention"] = l_ga
        out["loss"] = total
        return out

    # -- inference ---------------------------------------------------------
    @torch.no_grad()
    def infer(self, text: torch.Tensor, speaker: torch.Tensor | None = None, max_steps: int | None = None):
        """text (1, N) -> mel (1, n_mels, T), attention (T/r, N)."""
        text_lens = torch.tensor([text.size(1)], device=text.device)
        memory, processed, mask = self.encode(text, text_lens, speaker)
        state = self._init_state(memory)
        prev = memory.new_zeros(1, self.n_mels)
        mels, attns = [], []
        for _ in range(max_steps or self.cfg.model.max_decoder_steps):
            m_out, s_out, attn = self._step(prev, state, memory, processed, mask)
            group = m_out.view(1, self.r, self.n_mels)
            mels.append(group)
            attns.append(attn[0])
            prev = group[:, -1]
            if torch.sigmoid(s_out).max() > 0.5:
                break
        mel = torch.cat(mels, 1).transpose(1, 2)
        return mel + self.postnet(mel), torch.stack(attns)

### 5.2 Transformer-TTS (`src/tts/transformer_tts.py`)

Transformer-TTS (Li et al. 2019): Tacotron 2 with the RNNs and the location attention
replaced by a Transformer encoder-decoder.

```
text -> Embedding -> conv prenet -> + scaled positional encoding -> L x [self-attn, FFN]  (encoder)
mel frames (shifted right) -> PreNet -> + positional encoding
   -> L x [causal self-attn, cross-attn to the encoder, FFN]                           (decoder)
   -> Linear: r mel frames + stop logit ;  PostNet residual
```

Pros: every decoder step sees every earlier frame (no LSTM bottleneck) and training is
parallel over time (one causal mask instead of a Python loop). Cons: nothing forces the
cross-attention to be monotonic, so a diagonal *guided-attention loss* is important, and
inference is still frame-by-frame (slow for long sentences), which FastSpeech 2 fixes.

In [ ]:
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn



class DecoderLayer(nn.Module):
    def __init__(self, dim: int, heads: int, dropout: float):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(dim, dim * 4), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))
        self.ln1, self.ln2, self.ln3 = nn.LayerNorm(dim), nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, causal_mask, mem_mask):
        h = self.ln1(x)
        h, _ = self.self_attn(h, h, h, attn_mask=causal_mask, need_weights=False)
        x = x + self.drop(h)
        h = self.ln2(x)
        h, attn = self.cross_attn(h, memory, memory, key_padding_mask=~mem_mask, need_weights=True)
        x = x + self.drop(h)
        x = x + self.drop(self.ff(self.ln3(x)))
        return x, attn  # attn (B, T, N) averaged over heads


class TransformerTTS(nn.Module):
    def __init__(self, cfg: Config, vocab_size: int):
        super().__init__()
        m, a = cfg.model, cfg.audio
        self.cfg, self.n_mels, self.r = cfg, a.n_mels, m.reduction_factor
        H = m.hidden
        self.embed = nn.Embedding(vocab_size, H, padding_idx=0)
        self.enc_prenet = nn.Sequential(*[ConvNorm(H, H, 5, act=nn.ReLU(), dropout=m.dropout) for _ in range(3)])
        self.enc_pos, self.dec_pos = PositionalEncoding(H), PositionalEncoding(H)
        self.encoder = nn.ModuleList(FFTBlock(H, m.heads, 1, m.dropout) for _ in range(m.layers))
        self.speaker = SpeakerEmbedding(m.n_speakers, m.speaker_dim, H)
        self.dec_prenet = PreNet(a.n_mels, m.prenet_dim)
        self.dec_in = nn.Linear(m.prenet_dim, H)
        self.decoder = nn.ModuleList(DecoderLayer(H, m.heads, m.dropout) for _ in range(m.layers))
        self.ln = nn.LayerNorm(H)
        self.mel_out = nn.Linear(H, a.n_mels * self.r)
        self.stop_out = nn.Linear(H, self.r)
        self.postnet = PostNet(a.n_mels, m.postnet_channels, m.postnet_layers, dropout=m.dropout)

    def encode(self, text, text_lens, speaker):
        mask = sequence_mask(text_lens, text.size(1))
        x = self.enc_prenet(self.embed(text).transpose(1, 2)).transpose(1, 2)
        x = self.enc_pos(x)
        for block in self.encoder:
            x = block(x, mask)
        return self.speaker(x, speaker), mask

    def decode(self, frames_in, memory, mem_mask):
        """frames_in (B, S, n_mels): the frame that precedes each decoder step (zeros first)."""
        S = frames_in.size(1)
        x = self.dec_pos(self.dec_in(self.dec_prenet(frames_in)))
        causal = torch.triu(torch.ones(S, S, dtype=torch.bool, device=x.device), 1)
        attn = None
        for layer in self.decoder:
            x, attn = layer(x, memory, causal, mem_mask)
        x = self.ln(x)
        return self.mel_out(x), self.stop_out(x), attn

    def forward(self, batch: dict) -> dict:
        mel, _mel_lens, text_lens = batch["mel"], batch["mel_lens"], batch["text_lens"]
        B, _, T = mel.shape
        pad = (-T) % self.r
        frames = F.pad(mel, (0, pad)).transpose(1, 2)  # (B, T_pad, n_mels)
        groups = frames.view(B, -1, self.r, self.n_mels)  # (B, S, r, n_mels)
        frames_in = torch.cat([frames.new_zeros(B, 1, self.n_mels), groups[:, :-1, -1]], 1)
        memory, mem_mask = self.encode(batch["text"], text_lens, batch.get("speaker"))
        m_out, s_out, attn = self.decode(frames_in, memory, mem_mask)
        mel_before = m_out.view(B, -1, self.n_mels).transpose(1, 2)[:, :, :T]
        stop_logits = s_out.reshape(B, -1)[:, :T]
        mel_after = mel_before + self.postnet(mel_before)
        return self.loss(mel_before, mel_after, stop_logits, attn, batch) | {"mel": mel_after, "attention": attn}

    def loss(self, before, after, stop_logits, attn, batch) -> dict:
        mel, mel_lens, text_lens = batch["mel"], batch["mel_lens"], batch["text_lens"]
        mask = sequence_mask(mel_lens, mel.size(2))[:, None, :]
        n = mask.sum() * mel.size(1)
        l_before = (F.mse_loss(before, mel, reduction="none") * mask).sum() / n
        l_after = (F.mse_loss(after, mel, reduction="none") * mask).sum() / n
        stop_target = (torch.arange(mel.size(2), device=mel.device)[None] >= (mel_lens - 1)[:, None]).float()
        l_stop = F.binary_cross_entropy_with_logits(stop_logits, stop_target, pos_weight=torch.tensor(5.0))
        total = l_before + l_after + l_stop
        out = {"mel_before": l_before, "mel_after": l_after, "stop": l_stop}
        if self.cfg.train.guided_attention > 0:
            steps = torch.div(mel_lens + self.r - 1, self.r, rounding_mode="floor")
            l_ga = guided_attention_loss(attn, text_lens, steps)
            total = total + self.cfg.train.guided_attention * l_ga
            out["guided_attention"] = l_ga
        out["loss"] = total
        return out

    @torch.no_grad()
    def infer(self, text, speaker=None, max_steps=None):
        """Autoregressive: re-run the decoder on all frames so far at each step (O(T^2), simple)."""
        text_lens = torch.tensor([text.size(1)], device=text.device)
        memory, mem_mask = self.encode(text, text_lens, speaker)
        frames_in = memory.new_zeros(1, 1, self.n_mels)
        attn = None
        for _ in range(max_steps or self.cfg.model.max_decoder_steps):
            m_out, s_out, attn = self.decode(frames_in, memory, mem_mask)
            last = m_out[:, -1].view(1, self.r, self.n_mels)
            frames_in = torch.cat([frames_in, last[:, -1:]], 1)
            if torch.sigmoid(s_out[:, -1]).max() > 0.5:
                break
        mel = m_out.view(1, -1, self.n_mels).transpose(1, 2)
        return mel + self.postnet(mel), attn[0]

### 5.3 FastSpeech 2 (`src/tts/fastspeech2.py`)

FastSpeech 2 (Ren et al. 2020): non-autoregressive text -> mel, all frames at once.

```
text -> Embedding + positional encoding -> L x FFTBlock                    (encoder, (B, N, H))
     -> variance adaptor:
          duration predictor  d_n  (log frames per phoneme)      target: aligner / teacher
          pitch predictor     p_n  -> + pitch embedding            target: mean F0 of the phoneme's frames
          energy predictor    e_n  -> + energy embedding           target: mean energy of those frames
     -> LengthRegulator: repeat state n  d_n times                 (B, T, H)
     -> + positional encoding -> L x FFTBlock -> Linear -> mel     (decoder)
     -> + PostNet residual
```

There is no attention between text and frames, so the model *must* be told the durations.
By default a small **aligner** (see `align.py`) is trained jointly with the forward-sum
loss and gives hard durations through monotonic alignment search; `--duration-source
teacher` uses durations cached from a Tacotron 2 attention instead. Inference is one
forward pass (hundreds of times faster than an AR decoder), fully controllable (scale the
durations to speak slower, shift the pitch), and never loops or repeats words.

In [ ]:
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn



def average_by_duration(values: torch.Tensor, durations: torch.Tensor) -> torch.Tensor:
    """Frame-level (B, T) -> phoneme-level (B, N) mean over each phoneme's frames."""
    B, N = durations.shape
    out = values.new_zeros(B, N)
    for b in range(B):
        ends = durations[b].cumsum(0)
        starts = ends - durations[b]
        for n in range(N):
            if durations[b, n] > 0:
                seg = values[b, starts[n]:ends[n]]
                voiced = seg[seg > 0]
                out[b, n] = voiced.mean() if voiced.numel() else 0.0
    return out


class FastSpeech2(nn.Module):
    def __init__(self, cfg: Config, vocab_size: int):
        super().__init__()
        m, a = cfg.model, cfg.audio
        self.cfg = cfg
        H = m.hidden
        self.embed = nn.Embedding(vocab_size, H, padding_idx=0)
        self.enc_pos, self.dec_pos = PositionalEncoding(H), PositionalEncoding(H)
        self.encoder = nn.ModuleList(FFTBlock(H, m.heads, m.ffn_kernel, m.dropout) for _ in range(m.layers))
        self.decoder = nn.ModuleList(FFTBlock(H, m.heads, m.ffn_kernel, m.dropout) for _ in range(m.layers))
        self.speaker = SpeakerEmbedding(m.n_speakers, m.speaker_dim, H)
        self.duration_pred = VariancePredictor(H, dropout=m.dropout)
        self.pitch_pred = VariancePredictor(H, dropout=m.dropout)
        self.energy_pred = VariancePredictor(H, dropout=m.dropout)
        self.pitch_embed = nn.Conv1d(1, H, 9, padding=4)
        self.energy_embed = nn.Conv1d(1, H, 9, padding=4)
        self.length_regulator = LengthRegulator()
        self.mel_out = nn.Linear(H, a.n_mels)
        self.postnet = PostNet(a.n_mels, m.postnet_channels, m.postnet_layers, dropout=m.dropout)
        self.aligner = Aligner(H, a.n_mels) if cfg.train.duration_source == "aligner" else None

    def encode(self, text, text_lens, speaker):
        mask = sequence_mask(text_lens, text.size(1))
        x = self.enc_pos(self.embed(text))
        for block in self.encoder:
            x = block(x, mask)
        return self.speaker(x, speaker), mask

    def decode(self, x, durations):
        x, mel_lens = self.length_regulator(x, durations)
        mask = sequence_mask(mel_lens, x.size(1))
        x = self.dec_pos(x)
        for block in self.decoder:
            x = block(x, mask)
        mel = self.mel_out(x).transpose(1, 2)  # (B, n_mels, T)
        return mel, mel + self.postnet(mel), mel_lens

    def variance(self, x, mask, pitch=None, energy=None):
        """Predict log-duration, pitch, energy; add pitch/energy embeddings (targets if given)."""
        log_dur = self.duration_pred(x, mask)
        p = self.pitch_pred(x, mask)
        e = self.energy_pred(x, mask)
        x = x + self.pitch_embed((pitch if pitch is not None else p).unsqueeze(1)).transpose(1, 2)
        x = x + self.energy_embed((energy if energy is not None else e).unsqueeze(1)).transpose(1, 2)
        return x, log_dur, p, e

    def forward(self, batch: dict) -> dict:
        text, text_lens, mel, mel_lens = batch["text"], batch["text_lens"], batch["mel"], batch["mel_lens"]
        x, mask = self.encode(text, text_lens, batch.get("speaker"))
        out = {}
        if self.aligner is not None:
            log_probs = self.aligner(self.embed(text), mel, mask)  # (B, N, T)
            out["forward_sum"] = forward_sum_loss(log_probs, text_lens, mel_lens)
            hard = mas_batch(log_probs, text_lens, mel_lens)
            # binarisation loss: pull the soft alignment towards the hard path (after a warm-up it is ~0)
            out["bin"] = -(hard * log_probs).sum() / hard.sum()
            durations = durations_from_alignment(hard)
        else:
            durations = batch["durations"]
        pitch_t = average_by_duration(torch.log1p(batch["f0"]), durations)  # log-F0 per phoneme (0 = unvoiced)
        energy_t = average_by_duration(batch["energy"], durations)
        x, log_dur, p, e = self.variance(x, mask, pitch_t, energy_t)
        before, after, _ = self.decode(x, durations)
        T = mel.size(2)
        before, after = before[:, :, :T], after[:, :, :T]
        if before.size(2) < T:
            before, after = F.pad(before, (0, T - before.size(2))), F.pad(after, (0, T - after.size(2)))
        fmask = sequence_mask(mel_lens, T)[:, None, :]
        n = fmask.sum() * mel.size(1)
        out["mel_before"] = (F.l1_loss(before, mel, reduction="none") * fmask).sum() / n
        out["mel_after"] = (F.l1_loss(after, mel, reduction="none") * fmask).sum() / n
        out["duration"] = (F.mse_loss(log_dur, torch.log(durations.float() + 1), reduction="none") * mask).sum() / mask.sum()
        out["pitch"] = (F.mse_loss(p, pitch_t, reduction="none") * mask).sum() / mask.sum()
        out["energy"] = (F.mse_loss(e, energy_t, reduction="none") * mask).sum() / mask.sum()
        out["loss"] = sum(v for k, v in out.items())
        out["mel"], out["durations"] = after, durations
        if self.aligner is not None:
            out["attention"] = hard.transpose(1, 2)  # (B, T, N) like the AR models
        return out

    @torch.no_grad()
    def infer(self, text, speaker=None, duration_scale: float = 1.0):
        text_lens = torch.tensor([text.size(1)], device=text.device)
        x, mask = self.encode(text, text_lens, speaker)
        x, log_dur, _p, _e = self.variance(x, mask)
        durations = torch.clamp(torch.round((torch.exp(log_dur) - 1) * duration_scale), min=1).long() * mask
        _, mel, _ = self.decode(x, durations)
        hard = torch.repeat_interleave(torch.eye(text.size(1), device=text.device), durations[0], dim=0)
        return mel, hard  # (1, n_mels, T), (T, N)

## 6. Vocoders, GAN losses and VITS

### 6.1 HiFi-GAN generator and discriminators (`src/tts/vocoders.py`)

Vocoders: mel-spectrogram -> waveform.

- **Griffin-Lim** (`audio.griffin_lim`): no parameters, iterative phase recovery. Always
  available, robotic quality. The baseline and the vocoder of the unit tests.
- **HiFi-GAN** (Kong et al. 2020): a fully convolutional *generator* upsamples the mel
  (hop_length x) with transposed convolutions, each followed by a *multi-receptive-field
  fusion* block (sum of residual stacks with different kernel sizes / dilations).
  It is trained as a GAN against two families of *discriminators*:
  - MPD (multi-period): reshapes the waveform into 2-D (period x time) for periods
    2, 3, 5, 7, 11, so each sub-discriminator looks at a different periodic structure;
  - MSD (multi-scale): 1-D convolutions on the waveform, its 2x and 4x downsampled versions.
  Losses (`losses.py`): least-squares GAN, *feature matching* (L1 between the discriminators'
  hidden activations on real and fake), and a mel L1 reconstruction loss (weight 45).
  The same generator and discriminators are the waveform part of VITS.

In [ ]:
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils.parametrizations import weight_norm


LRELU = 0.1


class ResBlock(nn.Module):
    def __init__(self, channels: int, kernel: int, dilations: tuple[int, ...]):
        super().__init__()
        self.convs1 = nn.ModuleList(
            weight_norm(nn.Conv1d(channels, channels, kernel, dilation=d, padding=(kernel - 1) // 2 * d))
            for d in dilations)
        self.convs2 = nn.ModuleList(
            weight_norm(nn.Conv1d(channels, channels, kernel, padding=(kernel - 1) // 2)) for _ in dilations)

    def forward(self, x):
        for c1, c2 in zip(self.convs1, self.convs2):
            h = c1(F.leaky_relu(x, LRELU))
            x = x + c2(F.leaky_relu(h, LRELU))
        return x


class HiFiGANGenerator(nn.Module):
    """(B, cin, T) -> (B, prod(upsample_rates) * T) waveform in [-1, 1]."""

    def __init__(self, cfg: Config, cin: int | None = None):
        super().__init__()
        m = cfg.model
        cin = cin or cfg.audio.n_mels
        assert int(torch.tensor(m.upsample_rates).prod()) == cfg.audio.hop_length, \
            "product of upsample_rates must equal hop_length"
        ch = m.upsample_initial
        self.pre = weight_norm(nn.Conv1d(cin, ch, 7, padding=3))
        self.ups, self.mrfs = nn.ModuleList(), nn.ModuleList()
        for r in m.upsample_rates:
            self.ups.append(weight_norm(nn.ConvTranspose1d(ch, ch // 2, 2 * r, stride=r, padding=r // 2)))
            ch //= 2
            self.mrfs.append(nn.ModuleList(ResBlock(ch, k, m.resblock_dilations) for k in m.resblock_kernels))
        self.post = weight_norm(nn.Conv1d(ch, 1, 7, padding=3))

    def forward(self, x):
        x = self.pre(x)
        for up, mrf in zip(self.ups, self.mrfs):
            x = up(F.leaky_relu(x, LRELU))
            x = sum(block(x) for block in mrf) / len(mrf)
        return torch.tanh(self.post(F.leaky_relu(x, LRELU))).squeeze(1)


class PeriodDiscriminator(nn.Module):
    def __init__(self, period: int, channels: int = 32):
        super().__init__()
        self.period = period
        chs = [1, channels, channels * 2, channels * 4]
        self.convs = nn.ModuleList(
            weight_norm(nn.Conv2d(chs[i], chs[i + 1], (5, 1), stride=(3, 1), padding=(2, 0))) for i in range(3))
        self.post = weight_norm(nn.Conv2d(chs[-1], 1, (3, 1), padding=(1, 0)))

    def forward(self, wav):  # (B, samples)
        B, T = wav.shape
        pad = (-T) % self.period
        x = F.pad(wav, (0, pad), mode="reflect").view(B, 1, (T + pad) // self.period, self.period)
        feats = []
        for conv in self.convs:
            x = F.leaky_relu(conv(x), LRELU)
            feats.append(x)
        x = self.post(x)
        feats.append(x)
        return x.flatten(1), feats


class ScaleDiscriminator(nn.Module):
    def __init__(self, channels: int = 32):
        super().__init__()
        self.convs = nn.ModuleList([
            weight_norm(nn.Conv1d(1, channels, 15, padding=7)),
            weight_norm(nn.Conv1d(channels, channels * 2, 41, stride=4, groups=4, padding=20)),
            weight_norm(nn.Conv1d(channels * 2, channels * 4, 41, stride=4, groups=8, padding=20)),
            weight_norm(nn.Conv1d(channels * 4, channels * 4, 5, padding=2)),
        ])
        self.post = weight_norm(nn.Conv1d(channels * 4, 1, 3, padding=1))

    def forward(self, wav):
        x = wav.unsqueeze(1)
        feats = []
        for conv in self.convs:
            x = F.leaky_relu(conv(x), LRELU)
            feats.append(x)
        x = self.post(x)
        feats.append(x)
        return x.flatten(1), feats


class Discriminators(nn.Module):
    """MPD (periods 2, 3, 5, 7, 11) + MSD (scales 1, 2, 4). Returns lists of (logits, features)."""

    def __init__(self, periods=(2, 3, 5, 7, 11), scales: int = 3, channels: int = 16):
        super().__init__()
        self.periods = nn.ModuleList(PeriodDiscriminator(p, channels) for p in periods)
        self.scales = nn.ModuleList(ScaleDiscriminator(channels) for _ in range(scales))
        self.pool = nn.AvgPool1d(4, 2, padding=2)

    def forward(self, wav):
        outs, feats = [], []
        for d in self.periods:
            o, f = d(wav)
            outs.append(o)
            feats.append(f)
        x = wav
        for i, d in enumerate(self.scales):
            if i > 0:
                x = self.pool(x.unsqueeze(1)).squeeze(1)
            o, f = d(x)
            outs.append(o)
            feats.append(f)
        return outs, feats

### 6.2 GAN / KL losses (`src/tts/losses.py`)

GAN losses shared by HiFi-GAN and VITS (least-squares GAN, Mao et al. 2017).

- discriminator: (D(real) - 1)^2 + D(fake)^2        -> real towards 1, fake towards 0
- generator:     (D(fake) - 1)^2                      -> fool the discriminator
- feature matching: L1 between every hidden activation of D on real vs. fake; a smooth,
  perceptual reconstruction loss that keeps GAN training stable
- mel loss: L1 between log-mels of real and generated waveforms (weight 45 in HiFi-GAN)
- KL (VITS): KL( q(z | spectrogram) || p(z | text) ) between two diagonal Gaussians,
  computed on the flow-transformed posterior sample z_p

In [ ]:
from __future__ import annotations

import torch
import torch.nn.functional as F


def discriminator_loss(real_outs: list[torch.Tensor], fake_outs: list[torch.Tensor]) -> torch.Tensor:
    return sum(((r - 1) ** 2).mean() + (f**2).mean() for r, f in zip(real_outs, fake_outs))


def generator_loss(fake_outs: list[torch.Tensor]) -> torch.Tensor:
    return sum(((f - 1) ** 2).mean() for f in fake_outs)


def feature_matching_loss(real_feats: list[list[torch.Tensor]], fake_feats: list[list[torch.Tensor]]) -> torch.Tensor:
    total = 0.0
    for rf, ff in zip(real_feats, fake_feats):
        for r, f in zip(rf, ff):
            total = total + F.l1_loss(f, r.detach())
    return total


def kl_loss(z_p: torch.Tensor, logs_q: torch.Tensor, mu_p: torch.Tensor, logs_p: torch.Tensor,
            mask: torch.Tensor) -> torch.Tensor:
    """All (B, C, T); mask (B, 1, T). KL between N(z; mu_q, s_q) pushed through the flow and N(mu_p, s_p)."""
    kl = logs_p - logs_q - 0.5 + 0.5 * (z_p - mu_p) ** 2 * torch.exp(-2.0 * logs_p)
    return (kl * mask).sum() / mask.sum() / z_p.size(1)

### 6.3 VITS (`src/tts/vits.py`)

VITS (Kim et al. 2021), compact: end-to-end text -> waveform in one model.

VITS is a conditional VAE whose decoder is a HiFi-GAN generator, with a normalising flow
that makes the prior expressive and MAS that aligns text to frames, all trained together:

```
training
  linear spectrogram --PosteriorEncoder--> (mu_q, logs_q) --sample--> z      (B, C, T)
  z --Flow f--> z_p                                                       (B, C, T)
  text --TextEncoder--> (mu_p, logs_p)                                    (B, C, N)
  alignment = MAS( log N(z_p ; mu_p, logs_p) )          hard, monotonic   (B, N, T)
  expand mu_p, logs_p with the alignment to T frames
  KL loss   = KL( N(z_p; ., logs_q) || N(mu_p, logs_p) )
  duration  = MSE( DurationPredictor(text), log(alignment durations) )    # deterministic (see below)
  random slice of z --HiFiGANGenerator--> wav_hat;  mel L1 + GAN + feature matching vs the real slice

inference
  text -> (mu_p, logs_p), durations -> expand -> z_p = mu_p + noise * exp(logs_p) * temperature
       -> inverse flow -> z -> generator -> waveform
```

Simplification kept on purpose: the *stochastic* duration predictor of the paper (a flow
over durations) is replaced by the deterministic FastSpeech-style predictor; the rest of
the model (posterior encoder, coupling flow, MAS, GAN decoder) is the real thing.

In [ ]:
from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn



class WaveNetStack(nn.Module):
    """Dilated gated convolutions with residual + skip connections (the WN of WaveGlow/VITS)."""

    def __init__(self, channels: int, kernel: int = 5, layers: int = 4, dilation_rate: int = 2):
        super().__init__()
        self.in_layers = nn.ModuleList()
        self.res_skip = nn.ModuleList()
        for i in range(layers):
            d = dilation_rate**i
            self.in_layers.append(nn.Conv1d(channels, 2 * channels, kernel, dilation=d, padding=(kernel - 1) // 2 * d))
            self.res_skip.append(nn.Conv1d(channels, 2 * channels if i < layers - 1 else channels, 1))
        self.layers = layers

    def forward(self, x, mask):
        out = torch.zeros_like(x)
        for i in range(self.layers):
            a, b = self.in_layers[i](x).chunk(2, dim=1)
            h = torch.tanh(a) * torch.sigmoid(b)
            rs = self.res_skip[i](h)
            if i < self.layers - 1:
                res, skip = rs.chunk(2, dim=1)
                x = (x + res) * mask
                out = out + skip
            else:
                out = out + rs
        return out * mask


class PosteriorEncoder(nn.Module):
    """linear spectrogram (B, n_fft/2+1, T) -> z, mu_q, logs_q (B, C, T)."""

    def __init__(self, cin: int, hidden: int, latent: int):
        super().__init__()
        self.pre = nn.Conv1d(cin, hidden, 1)
        self.wn = WaveNetStack(hidden)
        self.proj = nn.Conv1d(hidden, 2 * latent, 1)

    def forward(self, spec, mask):
        h = self.wn(self.pre(spec) * mask, mask)
        mu, logs = self.proj(h).chunk(2, dim=1)
        z = (mu + torch.randn_like(mu) * torch.exp(logs)) * mask
        return z, mu * mask, logs * mask


class CouplingLayer(nn.Module):
    """Affine coupling with mean-only transform (as in VITS): x = [x0, x1]; x1' = x1 + m(x0)."""

    def __init__(self, latent: int, hidden: int):
        super().__init__()
        self.half = latent // 2
        self.pre = nn.Conv1d(self.half, hidden, 1)
        self.wn = WaveNetStack(hidden, layers=3)
        self.post = nn.Conv1d(hidden, self.half, 1)
        nn.init.zeros_(self.post.weight)
        nn.init.zeros_(self.post.bias)

    def forward(self, x, mask, reverse=False):
        x0, x1 = x[:, :self.half], x[:, self.half:]
        m = self.post(self.wn(self.pre(x0) * mask, mask)) * mask
        x1 = x1 - m if reverse else x1 + m
        return torch.cat([x0, x1], dim=1)


class Flow(nn.Module):
    """Stack of coupling layers, flipping channel halves between them (volume preserving)."""

    def __init__(self, latent: int, hidden: int, blocks: int):
        super().__init__()
        self.layers = nn.ModuleList(CouplingLayer(latent, hidden) for _ in range(blocks))

    def forward(self, x, mask, reverse=False):
        layers = reversed(self.layers) if reverse else self.layers
        for layer in layers:
            if reverse:
                x = torch.flip(x, [1])
                x = layer(x, mask, reverse=True)
            else:
                x = layer(x, mask)
                x = torch.flip(x, [1])
        return x


class VITS(nn.Module):
    def __init__(self, cfg: Config, vocab_size: int):
        super().__init__()
        m, a = cfg.model, cfg.audio
        self.cfg = cfg
        H, C = m.hidden, m.latent_dim
        self.embed = nn.Embedding(vocab_size, H, padding_idx=0)
        self.pos = PositionalEncoding(H)
        self.encoder = nn.ModuleList(FFTBlock(H, m.heads, m.ffn_kernel, m.dropout) for _ in range(m.layers))
        self.speaker = SpeakerEmbedding(m.n_speakers, m.speaker_dim, H)
        self.proj = nn.Conv1d(H, 2 * C, 1)
        self.duration_pred = VariancePredictor(H, dropout=m.dropout)
        self.posterior = PosteriorEncoder(a.n_fft // 2 + 1, H, C)
        self.flow = Flow(C, H, m.flow_blocks)
        self.decoder = HiFiGANGenerator(cfg, cin=C)
        self.segment_frames = m.segment_frames

    def encode_text(self, text, text_lens, speaker):
        mask = sequence_mask(text_lens, text.size(1))
        x = self.pos(self.embed(text))
        for block in self.encoder:
            x = block(x, mask)
        x = self.speaker(x, speaker)
        mu_p, logs_p = self.proj(x.transpose(1, 2)).chunk(2, dim=1)  # (B, C, N)
        return x, mu_p * mask[:, None], logs_p * mask[:, None], mask

    @staticmethod
    def random_slice(z, lengths, seg):
        """Pick one segment of `seg` frames per item (within its length). Returns slice and start indices."""
        B = z.size(0)
        starts = [int(torch.randint(0, max(int(lengths[b]) - seg, 0) + 1, (1,))) for b in range(B)]
        return torch.stack([z[b, :, s:s + seg] for b, s in enumerate(starts)]), starts

    def forward(self, batch: dict) -> dict:
        """Returns z-slice decoded waveform, KL/duration losses and the slice starts (train.py adds the GAN)."""
        text, text_lens, spec, mel_lens = batch["text"], batch["text_lens"], batch["linear"], batch["mel_lens"]
        x, mu_p, logs_p, text_mask = self.encode_text(text, text_lens, batch.get("speaker"))
        spec_mask = sequence_mask(mel_lens, spec.size(2))[:, None]  # (B, 1, T)
        z, _mu_q, logs_q = self.posterior(spec, spec_mask)
        z_p = self.flow(z, spec_mask)
        with torch.no_grad():  # log N(z_p ; mu_p, logs_p) for every (text n, frame t): (B, N, T)
            s_p_sq_r = torch.exp(-2 * logs_p)  # (B, C, N)
            neg_cent1 = torch.sum(-0.5 * math.log(2 * math.pi) - logs_p, 1, keepdim=True).transpose(1, 2)  # (B,N,1)
            neg_cent2 = torch.matmul((-0.5 * s_p_sq_r).transpose(1, 2), z_p**2)  # (B, N, T)
            neg_cent3 = torch.matmul((mu_p * s_p_sq_r).transpose(1, 2), z_p)
            neg_cent4 = torch.sum(-0.5 * mu_p**2 * s_p_sq_r, 1, keepdim=True).transpose(1, 2)
            logp = neg_cent1 + neg_cent2 + neg_cent3 + neg_cent4
            logp = logp.masked_fill(~(text_mask[:, :, None] & spec_mask), -1e9)
            attn = mas_batch(logp, text_lens, mel_lens)  # (B, N, T)
        durations = attn.sum(2)
        log_dur = self.duration_pred(x, text_mask)
        l_dur = (F.mse_loss(log_dur, torch.log(durations + 1), reduction="none") * text_mask).sum() / text_mask.sum()
        mu_p_t = torch.matmul(mu_p, attn)  # (B, C, T): expand to frames
        logs_p_t = torch.matmul(logs_p, attn)
        kl = logs_p_t - logs_q - 0.5 + 0.5 * (z_p - mu_p_t) ** 2 * torch.exp(-2.0 * logs_p_t)
        l_kl = (kl * spec_mask).sum() / spec_mask.sum() / z.size(1)
        z_slice, starts = self.random_slice(z, mel_lens, self.segment_frames)
        wav_hat = self.decoder(z_slice)
        return {"wav_hat": wav_hat, "starts": starts, "kl": l_kl, "duration": l_dur, "attention": attn.transpose(1, 2),
                "durations": durations.long()}

    @torch.no_grad()
    def infer(self, text, speaker=None, duration_scale: float = 1.0, temperature: float = 0.667):
        text_lens = torch.tensor([text.size(1)], device=text.device)
        x, mu_p, logs_p, mask = self.encode_text(text, text_lens, speaker)
        log_dur = self.duration_pred(x, mask)
        durations = torch.clamp(torch.round((torch.exp(log_dur) - 1) * duration_scale), min=1).long() * mask
        attn = torch.repeat_interleave(torch.eye(text.size(1), device=text.device), durations[0], dim=0)  # (T, N)
        mu_p_t, logs_p_t = mu_p[0] @ attn.T, logs_p[0] @ attn.T  # (C, T)
        z_p = (mu_p_t + torch.randn_like(mu_p_t) * torch.exp(logs_p_t) * temperature)[None]
        ones = torch.ones(1, 1, z_p.size(2), device=text.device)
        z = self.flow(z_p, ones, reverse=True)
        return self.decoder(z)[0], attn  # (samples,), (T, N)

## 7. Data

### 7.1 Dataset and batching (`src/tts/data.py`)

Datasets and batching.

Every dataset is a folder in the **LJSpeech layout**:

```
<data_dir>/metadata.csv      lines "<id>|<text>" or "<id>|<raw text>|<normalised text>"
<data_dir>/wavs/<id>.wav
<data_dir>/speakers.csv      optional: "<id>|<speaker name>" for multi-speaker data
```

or a `manifest.jsonl` with `{"audio": ..., "text": ..., "speaker": ...}` per line. `tts-data`
writes the built-in datasets in this layout.

`TTSDataset` computes the log-mel, F0 and energy of each clip once and caches them as
`.pt` files under `<data_dir>/cache/`, so epochs after the first only read tensors.
A batch is a dict of padded tensors:
`text (B, N)`, `text_lens`, `mel (B, n_mels, T)`, `mel_lens`, `f0 (B, T)`, `energy (B, T)`,
`wav (B, samples)`, `speaker (B,)`, plus `linear (B, n_fft/2+1, T)` for VITS and
`durations (B, N)` when teacher durations exist in `<data_dir>/durations/`.

In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset



@dataclass
class Item:
    id: str
    audio: Path
    text: str
    speaker: str = "0"


def read_manifest(data_dir: str | Path) -> list[Item]:
    data_dir = Path(data_dir)
    items: list[Item] = []
    if (data_dir / "manifest.jsonl").exists():
        for line in (data_dir / "manifest.jsonl").read_text().splitlines():
            if line.strip():
                d = json.loads(line)
                audio = Path(d["audio"])
                audio = audio if audio.is_absolute() else data_dir / audio
                items.append(Item(d.get("id", audio.stem), audio, d["text"], str(d.get("speaker", "0"))))
        return items
    meta = data_dir / "metadata.csv"
    if not meta.exists():
        raise FileNotFoundError(f"no metadata.csv or manifest.jsonl in {data_dir}; run tts-data first")
    speakers: dict[str, str] = {}
    if (data_dir / "speakers.csv").exists():
        for line in (data_dir / "speakers.csv").read_text().splitlines():
            if "|" in line:
                k, v = line.split("|", 1)
                speakers[k] = v
    for line in meta.read_text(encoding="utf-8").splitlines():
        parts = line.split("|")
        if len(parts) < 2:
            continue
        text = parts[2] if len(parts) > 2 and parts[2].strip() else parts[1]
        items.append(Item(parts[0], data_dir / "wavs" / f"{parts[0]}.wav", text.strip(), speakers.get(parts[0], "0")))
    return items


def split_items(items: list[Item], val_fraction: float = 0.05, seed: int = 0) -> tuple[list[Item], list[Item]]:
    """Deterministic train/val split (at least 2 validation items)."""
    rng = random.Random(seed)
    order = list(items)
    rng.shuffle(order)
    n_val = max(2, int(len(order) * val_fraction)) if len(order) > 4 else 1
    return order[n_val:], order[:n_val]


class TTSDataset(Dataset):
    def __init__(self, items: list[Item], audio: AudioConfig, tokenizer: Tokenizer, data_dir: str | Path,
                 speaker_ids: dict[str, int] | None = None, need_linear: bool = False, need_wav: bool = False,
                 max_frames: int = 0):
        self.items, self.audio, self.tok = items, audio, tokenizer
        self.mel = MelSpectrogram(audio)
        self.cache = Path(data_dir) / "cache"
        self.cache.mkdir(parents=True, exist_ok=True)
        self.durations_dir = Path(data_dir) / "durations"
        self.speaker_ids = speaker_ids or {s: i for i, s in enumerate(sorted({it.speaker for it in items}))}
        self.need_linear, self.need_wav, self.max_frames = need_linear, need_wav, max_frames
        if max_frames:
            self.items = [it for it in self.items if self._n_frames(it) <= max_frames]

    def _n_frames(self, item: Item) -> int:
        return self._features(item)["mel"].size(1)

    def _features(self, item: Item) -> dict:
        tag = f"{self.audio.sample_rate}_{self.audio.n_fft}_{self.audio.hop_length}_{self.audio.n_mels}"
        path = self.cache / f"{item.id}_{tag}.pt"
        if path.exists():
            return torch.load(path)
        wav = load_wav(str(item.audio), self.audio.sample_rate)
        wav_t = torch.from_numpy(wav)
        mel = self.mel(wav_t[None])[0]
        f0 = torch.from_numpy(estimate_f0(wav, self.audio))[: mel.size(1)]
        f0 = torch.nn.functional.pad(f0, (0, mel.size(1) - f0.numel()))
        feats = {"mel": mel, "f0": f0, "energy": frame_energy(mel), "wav": wav_t}
        torch.save(feats, path)
        return feats

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, i: int) -> dict:
        item = self.items[i]
        feats = self._features(item)
        out = {"id": item.id, "text": torch.tensor(self.tok.encode(item.text)), "mel": feats["mel"], "f0": feats["f0"],
               "energy": feats["energy"], "speaker": self.speaker_ids.get(item.speaker, 0)}
        if self.need_wav or self.need_linear:
            out["wav"] = feats["wav"]
        if self.need_linear:
            out["linear"] = self.mel.linear(feats["wav"][None])[0]
        dur = self.durations_dir / f"{item.id}.pt"
        if dur.exists():
            out["durations"] = torch.load(dur)
        return out


def collate(samples: list[dict]) -> dict:
    """Pad every sequence in the batch to the longest one."""
    B = len(samples)
    N = max(s["text"].numel() for s in samples)
    T = max(s["mel"].size(1) for s in samples)
    n_mels = samples[0]["mel"].size(0)
    batch = {"ids": [s["id"] for s in samples], "text": torch.zeros(B, N, dtype=torch.long),
             "text_lens": torch.tensor([s["text"].numel() for s in samples]),
             "mel": torch.full((B, n_mels, T), float(samples[0]["mel"].min())),
             "mel_lens": torch.tensor([s["mel"].size(1) for s in samples]),
             "f0": torch.zeros(B, T), "energy": torch.zeros(B, T),
             "speaker": torch.tensor([s["speaker"] for s in samples])}
    for b, s in enumerate(samples):
        n, t = s["text"].numel(), s["mel"].size(1)
        batch["text"][b, :n] = s["text"]
        batch["mel"][b, :, :t] = s["mel"]
        batch["f0"][b, :t], batch["energy"][b, :t] = s["f0"][:t], s["energy"][:t]
    if "wav" in samples[0]:
        S = max(s["wav"].numel() for s in samples)
        batch["wav"] = torch.zeros(B, S)
        for b, s in enumerate(samples):
            batch["wav"][b, : s["wav"].numel()] = s["wav"]
    if "linear" in samples[0]:
        batch["linear"] = torch.zeros(B, samples[0]["linear"].size(0), T)
        for b, s in enumerate(samples):
            batch["linear"][b, :, : s["linear"].size(1)] = s["linear"][:, :T]
    if all("durations" in s for s in samples):
        batch["durations"] = torch.zeros(B, N, dtype=torch.long)
        for b, s in enumerate(samples):
            d = s["durations"][:N]
            batch["durations"][b, : d.numel()] = d
    return batch


def make_loader(dataset: TTSDataset, batch_size: int, shuffle: bool, num_workers: int = 0) -> DataLoader:
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate, num_workers=num_workers,
                      drop_last=False)

### 7.2 Built-in datasets (`src/tts/datasets.py`)

Built-in datasets, written in the LJSpeech layout by `tts-data <name> --out <dir>`.

- `synthetic`: a deterministic **toy spoken language** at 8 kHz. Every letter is a short
  harmonic tone with its own fundamental frequency and spectral envelope (a crude vowel),
  spaces are silences, and 25 dB of noise is added. The text -> audio mapping is simple and
  fully known, so a model that learns it proves the pipeline (alignment, durations, pitch,
  vocoder) works before any real data is touched. Used by the tests and the smoke runs.
- `fsdd`: Free Spoken Digit Dataset (3,000 recordings of "zero".."nine" by 6 speakers at
  8 kHz, ~ 20 MB). Real, low-quality speech; `--speaker jackson` (default) keeps one voice.
- `yesno`: 60 recordings of eight-word yes/no sequences at 8 kHz (openslr 1, 4.5 MB). The
  words are the Hebrew "ken"/"lo"; the transcripts use "yes"/"no".
- `ljspeech`: the standard 24 h single-speaker English corpus at 22.05 kHz (2.6 GB); for
  Kaggle/Colab. Use `--sample-rate 22050 --n-fft 1024 --hop-length 256 --win-length 1024
  --n-mels 80 --fmax 8000` with it.
- VCTK and any other corpus: write `metadata.csv` + `wavs/` (+ `speakers.csv`) yourself,
  or a `manifest.jsonl`.

In [ ]:
from __future__ import annotations

import argparse
import math
import random
import tarfile
import urllib.request
import zipfile
from pathlib import Path

import numpy as np


FSDD_URL = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/heads/master.zip"
YESNO_URL = "https://www.openslr.org/resources/1/waves_yesno.tar.gz"
LJSPEECH_URL = "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"
DIGITS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]

TOY_WORDS = ["ba", "ko", "mi", "tu", "sel", "ran", "dop", "lia", "nef", "goz", "we", "pyx", "hu", "qim", "jav"]


def _letter_profile(ch: str) -> tuple[float, list[float], float]:
    """(fundamental Hz, harmonic amplitudes, duration s) for one letter, deterministic."""
    i = ord(ch) - ord("a")
    f0 = 110.0 * 2 ** (i / 12.0)  # one semitone per letter: 110 Hz (a) .. 467 Hz (z)
    rng = random.Random(1000 + i)
    harmonics = [1.0] + [rng.uniform(0.1, 0.8) / (k + 1) for k in range(1, 5)]
    duration = 0.06 + 0.04 * ((i * 7) % 5) / 4  # 60..100 ms
    return f0, harmonics, duration


def synth_toy_wave(text: str, sample_rate: int = 8000, noise_db: float = -25.0, seed: int = 0) -> np.ndarray:
    """Render text in the toy language: letters -> harmonic tones, spaces -> 80 ms silence."""
    rng = np.random.default_rng(seed)
    pieces = [np.zeros(int(0.05 * sample_rate), dtype=np.float32)]
    for ch in text.lower():
        if ch == " ":
            pieces.append(np.zeros(int(0.08 * sample_rate), dtype=np.float32))
            continue
        if not ch.isalpha():
            continue
        f0, harmonics, dur = _letter_profile(ch)
        n = int(dur * sample_rate)
        t = np.arange(n) / sample_rate
        glide = 1.0 + 0.03 * np.sin(2 * math.pi * 3 * t)  # slight vibrato so F0 tracking is not trivial
        wave = sum(a * np.sin(2 * math.pi * f0 * (k + 1) * np.cumsum(glide) / sample_rate)
                   for k, a in enumerate(harmonics))
        env = np.minimum(1.0, np.minimum(t / 0.01, (dur - t) / 0.015))  # attack / release
        pieces.append((0.4 * wave * np.clip(env, 0, 1)).astype(np.float32))
    pieces.append(np.zeros(int(0.05 * sample_rate), dtype=np.float32))
    wav = np.concatenate(pieces)
    wav = wav + rng.normal(0, 10 ** (noise_db / 20), wav.shape).astype(np.float32)
    return np.clip(wav, -1, 1)


def toy_sentence(rng: random.Random, min_words: int = 2, max_words: int = 5) -> str:
    return " ".join(rng.choice(TOY_WORDS) for _ in range(rng.randint(min_words, max_words)))


def write_synthetic(out: Path, n: int = 300, sample_rate: int = 8000, seed: int = 0, n_speakers: int = 1) -> Path:
    """`n` toy sentences. With n_speakers > 1 every speaker shifts all pitches (a crude voice)."""
    out.mkdir(parents=True, exist_ok=True)
    (out / "wavs").mkdir(exist_ok=True)
    rng = random.Random(seed)
    lines, speakers = [], []
    for i in range(n):
        text = toy_sentence(rng)
        spk = i % n_speakers
        wav = synth_toy_wave(text, sample_rate, seed=seed + i)
        if spk:
            wav = _pitch_shift(wav, 2 ** (spk / 12.0 * 3))
        save_wav(str(out / "wavs" / f"toy{i:04d}.wav"), wav, sample_rate)
        lines.append(f"toy{i:04d}|{text}")
        speakers.append(f"toy{i:04d}|spk{spk}")
    (out / "metadata.csv").write_text("\n".join(lines) + "\n")
    if n_speakers > 1:
        (out / "speakers.csv").write_text("\n".join(speakers) + "\n")
    return out


def _pitch_shift(wav: np.ndarray, factor: float) -> np.ndarray:
    """Crude resampling pitch shift (changes duration too; fine for a toy)."""
    idx = np.arange(0, len(wav), factor)
    return np.interp(idx, np.arange(len(wav)), wav).astype(np.float32)


def _download(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        print(f"downloading {url} -> {dest}")
        urllib.request.urlretrieve(url, dest)
    return dest


def write_fsdd(out: Path, speaker: str | None = "jackson", cache: Path | None = None) -> Path:
    """Free Spoken Digit Dataset; `speaker=None` keeps all six speakers (writes speakers.csv)."""
    zip_path = _download(FSDD_URL, (cache or out) / "fsdd-master.zip")
    out.mkdir(parents=True, exist_ok=True)
    (out / "wavs").mkdir(exist_ok=True)
    lines, speakers = [], []
    with zipfile.ZipFile(zip_path) as zf:
        for name in sorted(zf.namelist()):
            if "/recordings/" not in name or not name.endswith(".wav"):
                continue
            stem = Path(name).stem  # <digit>_<speaker>_<index>
            digit, spk, _ = stem.split("_", 2)
            if speaker and spk != speaker:
                continue
            (out / "wavs" / f"{stem}.wav").write_bytes(zf.read(name))
            lines.append(f"{stem}|{DIGITS[int(digit)]}")
            speakers.append(f"{stem}|{spk}")
    (out / "metadata.csv").write_text("\n".join(lines) + "\n")
    if not speaker:
        (out / "speakers.csv").write_text("\n".join(speakers) + "\n")
    print(f"fsdd: {len(lines)} clips -> {out}")
    return out


def write_yesno(out: Path, cache: Path | None = None) -> Path:
    tar_path = _download(YESNO_URL, (cache or out) / "waves_yesno.tar.gz")
    out.mkdir(parents=True, exist_ok=True)
    (out / "wavs").mkdir(exist_ok=True)
    lines = []
    with tarfile.open(tar_path) as tf:
        for member in tf.getmembers():
            if not member.name.endswith(".wav"):
                continue
            stem = Path(member.name).stem
            data = tf.extractfile(member).read()
            (out / "wavs" / f"{stem}.wav").write_bytes(data)
            lines.append(f"{stem}|" + " ".join("yes" if c == "1" else "no" for c in stem.split("_")))
    (out / "metadata.csv").write_text("\n".join(sorted(lines)) + "\n")
    print(f"yesno: {len(lines)} clips -> {out}")
    return out


def write_ljspeech(out: Path, cache: Path | None = None) -> Path:
    """Downloads 2.6 GB and extracts LJSpeech-1.1 (already in the right layout) into `out`."""
    tar_path = _download(LJSPEECH_URL, (cache or out.parent) / "LJSpeech-1.1.tar.bz2")
    out.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:bz2") as tf:
        tf.extractall(out.parent)
    src = out.parent / "LJSpeech-1.1"
    if src != out:
        src.rename(out)
    print(f"ljspeech -> {out}")
    return out


def main(argv: list[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Write a built-in dataset in the LJSpeech layout.")
    p.add_argument("name", choices=["synthetic", "fsdd", "yesno", "ljspeech"])
    p.add_argument("--out", default=None, help="default: data/<name>")
    p.add_argument("--n", type=int, default=300, help="synthetic: number of sentences")
    p.add_argument("--n-speakers", type=int, default=1, help="synthetic: number of toy speakers")
    p.add_argument("--sample-rate", type=int, default=8000, help="synthetic only")
    p.add_argument("--speaker", default="jackson", help="fsdd: speaker to keep, 'all' for every speaker")
    p.add_argument("--seed", type=int, default=0)
    a = p.parse_args(argv)
    out = Path(a.out or f"data/{a.name}")
    if a.name == "synthetic":
        write_synthetic(out, a.n, a.sample_rate, a.seed, a.n_speakers)
        print(f"synthetic: {a.n} clips -> {out}")
    elif a.name == "fsdd":
        write_fsdd(out, None if a.speaker == "all" else a.speaker)
    elif a.name == "yesno":
        write_yesno(out)
    else:
        write_ljspeech(out)

In [ ]:
if SMOKE_TEST:
    DATA_DIR = write_synthetic(Path("data/synthetic"), n=200, sample_rate=AUDIO["sample_rate"])
elif DOWNLOAD_DATASET == "ljspeech":
    DATA_DIR = write_ljspeech(Path("data/ljspeech"))
elif DOWNLOAD_DATASET == "fsdd":
    DATA_DIR = write_fsdd(Path("data/fsdd"))
elif DOWNLOAD_DATASET == "yesno":
    DATA_DIR = write_yesno(Path("data/yesno"))
else:
    DATA_DIR = Path(DATA_DIR) if DATA_DIR else find_data_dir()
assert DATA_DIR is not None, "no dataset found: set DATA_DIR or DOWNLOAD_DATASET"
items = read_manifest(DATA_DIR)
print(f"{len(items)} clips in {DATA_DIR}; e.g. {items[0].id}: {items[0].text!r}")

## 8. Training, synthesis and metrics code

### 8.1 Model registry and checkpoints (`src/tts/models.py`)

Model registry and checkpoints.

A checkpoint is one `.pt` file holding the config, the tokenizer kind, the speaker map and
the weights (plus the discriminators and optimiser states for resuming), so `tts-synthesize`
and `tts-eval` need nothing but the file.

In [ ]:
from __future__ import annotations

from pathlib import Path

import torch
from torch import nn



def build_model(cfg: Config, vocab_size: int) -> nn.Module:
    name = cfg.model.model
    if name == "tacotron2":
        return Tacotron2(cfg, vocab_size)
    if name == "transformer_tts":
        return TransformerTTS(cfg, vocab_size)
    if name == "fastspeech2":
        return FastSpeech2(cfg, vocab_size)
    if name == "vits":
        return VITS(cfg, vocab_size)
    if name == "hifigan":
        return HiFiGANGenerator(cfg)
    raise ValueError(f"unknown model {name!r}")


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def save_checkpoint(path: str | Path, model: nn.Module, cfg: Config, speaker_ids: dict[str, int], step: int,
                    epoch: int, extra: dict | None = None) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    state = {"cfg": cfg.to_dict(), "model": model.state_dict(), "speaker_ids": speaker_ids, "step": step,
             "epoch": epoch}
    state.update(extra or {})
    torch.save(state, path)


def load_checkpoint(path: str | Path, device: str = "cpu") -> tuple[nn.Module, Config, Tokenizer, dict]:
    """-> (model in eval mode, config, tokenizer, raw checkpoint dict)."""
    state = torch.load(path, map_location=device, weights_only=False)
    cfg = Config.from_dict(state["cfg"])
    tok = Tokenizer(cfg.text.tokenizer, cfg.text.lowercase)
    model = build_model(cfg, tok.vocab_size)
    model.load_state_dict(state["model"])
    model.to(device).eval()
    return model, cfg, tok, state

### 8.2 Trainer (`src/tts/train.py`)

Training: one loop per family of models.

- **mel models** (tacotron2, transformer_tts, fastspeech2): one AdamW optimiser, the model's
  own `forward(batch)` returns every loss term; linear warm-up then inverse-sqrt decay.
- **hifigan**: random `segment_frames`-frame slices of the mel and the matching waveform;
  generator and discriminators alternate (D step on the detached fake, then G step with
  LS-GAN + feature matching + 45 x mel L1).
- **vits**: the model returns the decoded slice of z; the GAN part is the same as HiFi-GAN,
  plus the KL and duration losses.

Every run writes `last.pt`, `best.pt` (lowest validation mel loss), `curves.csv`, `curves.json`,
`config.json` and, when matplotlib is installed, `alignment_<step>.png` for the first
validation item, to `--output-dir`. `--resume` continues from `last.pt`; `--time-budget-hours`
saves and stops before a Kaggle/Colab session ends.

In [ ]:
from __future__ import annotations

import argparse
import csv
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F


GAN_MODELS = ("hifigan", "vits")
MEL_WEIGHT = 45.0


def pick_device(name: str) -> torch.device:
    if name != "auto":
        return torch.device(name)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def lr_at(step: int, base: float, warmup: int) -> float:
    """Linear warm-up, then 1/sqrt(step) decay (the Transformer schedule), floor at 5% of base."""
    if step < warmup:
        return base * (step + 1) / max(warmup, 1)
    return max(base * math.sqrt(warmup / (step + 1)), 0.05 * base)


def to_device(batch: dict, device: torch.device) -> dict:
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


def plot_alignment(attn: torch.Tensor, path: Path, title: str = "") -> None:
    """attn (T, N) -> png; silently skipped without matplotlib."""
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.imshow(attn.detach().cpu().numpy().T, aspect="auto", origin="lower", interpolation="none")
    ax.set_xlabel("decoder frames")
    ax.set_ylabel("text symbols")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=100)
    plt.close(fig)


class Trainer:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.device = pick_device(cfg.train.device)
        set_seed(cfg.train.seed)
        self.out = Path(cfg.train.output_dir)
        self.out.mkdir(parents=True, exist_ok=True)
        self.tok = Tokenizer(cfg.text.tokenizer, cfg.text.lowercase)
        self.mel_fn = MelSpectrogram(cfg.audio)
        self.gan = cfg.model.model in GAN_MODELS
        items = read_manifest(cfg.train.data_dir)
        train_items, val_items = split_items(items, seed=cfg.train.seed)
        speaker_ids = {s: i for i, s in enumerate(sorted({it.speaker for it in items}))}
        if len(speaker_ids) > 1 and cfg.model.n_speakers < len(speaker_ids):
            cfg.model.n_speakers = len(speaker_ids)
        kw = {"need_linear": cfg.model.model == "vits", "need_wav": self.gan, "max_frames": cfg.train.max_frames}
        self.train_set = TTSDataset(train_items, cfg.audio, self.tok, cfg.train.data_dir, speaker_ids, **kw)
        self.val_set = TTSDataset(val_items, cfg.audio, self.tok, cfg.train.data_dir, speaker_ids, **kw)
        self.speaker_ids = speaker_ids
        self.train_loader = make_loader(self.train_set, cfg.train.batch_size, True, cfg.train.num_workers)
        self.val_loader = make_loader(self.val_set, cfg.train.batch_size, False, cfg.train.num_workers)
        self.model = build_model(cfg, self.tok.vocab_size).to(self.device)
        self.disc = Discriminators().to(self.device) if self.gan else None
        t = cfg.train
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=t.lr, betas=(0.8, 0.99) if self.gan else (0.9, 0.98),
                                     weight_decay=t.weight_decay)
        self.opt_d = torch.optim.AdamW(self.disc.parameters(), lr=t.lr, betas=(0.8, 0.99),
                                       weight_decay=t.weight_decay) if self.gan else None
        self.scaler = torch.amp.GradScaler(enabled=t.amp and self.device.type == "cuda")
        self.step, self.epoch, self.best = 0, 0, float("inf")
        self.curves: list[dict] = []
        if t.resume and (self.out / "last.pt").exists():
            self.load(self.out / "last.pt")
        cfg.save(self.out / "config.json")
        print(f"{cfg.model.model}: {count_parameters(self.model) / 1e6:.2f} M parameters | "
              f"{len(self.train_set)} train / {len(self.val_set)} val clips | device {self.device}")

    # -- checkpoints --------------------------------------------------------
    def save(self, name: str) -> None:
        extra = {"opt": self.opt.state_dict(), "best": self.best, "curves": self.curves}
        if self.gan:
            extra["disc"], extra["opt_d"] = self.disc.state_dict(), self.opt_d.state_dict()
        save_checkpoint(self.out / name, self.model, self.cfg, self.speaker_ids, self.step, self.epoch, extra)

    def load(self, path: Path) -> None:
        state = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(state["model"])
        self.opt.load_state_dict(state["opt"])
        if self.gan and "disc" in state:
            self.disc.load_state_dict(state["disc"])
            self.opt_d.load_state_dict(state["opt_d"])
        self.step, self.epoch, self.best = state["step"], state["epoch"], state.get("best", float("inf"))
        self.curves = state.get("curves", [])
        print(f"resumed from {path} at step {self.step}")

    # -- one optimisation step ---------------------------------------------
    def gan_step(self, batch: dict) -> dict:
        seg = self.cfg.model.segment_frames
        hop = self.cfg.audio.hop_length
        if self.cfg.model.model == "vits":
            out = self.model(batch)
            wav_hat, starts = out["wav_hat"], out["starts"]
            losses = {"kl": out["kl"], "duration": out["duration"]}
        else:
            mel, lens = batch["mel"], batch["mel_lens"]
            starts = [int(torch.randint(0, max(int(lens[b]) - seg, 0) + 1, (1,))) for b in range(mel.size(0))]
            mel_slice = torch.stack([mel[b, :, s:s + seg] for b, s in enumerate(starts)])
            wav_hat = self.model(mel_slice)
            losses = {}
        wav = torch.stack([F.pad(batch["wav"][b, s * hop:(s + seg) * hop], (0, max(0, (s + seg) * hop - batch["wav"].size(1))))
                           for b, s in enumerate(starts)])
        wav_hat = wav_hat[:, : wav.size(1)]
        # discriminator
        self.opt_d.zero_grad(set_to_none=True)
        real_out, _ = self.disc(wav)
        fake_out, _ = self.disc(wav_hat.detach())
        l_d = discriminator_loss(real_out, fake_out)
        l_d.backward()
        torch.nn.utils.clip_grad_norm_(self.disc.parameters(), self.cfg.train.grad_clip)
        self.opt_d.step()
        # generator
        self.opt.zero_grad(set_to_none=True)
        real_out, real_feats = self.disc(wav)
        fake_out, fake_feats = self.disc(wav_hat)
        l_mel = F.l1_loss(self.mel_fn(wav_hat), self.mel_fn(wav))
        l_g = generator_loss(fake_out)
        l_fm = feature_matching_loss(real_feats, fake_feats)
        total = l_g + 2.0 * l_fm + MEL_WEIGHT * l_mel + sum(losses.values())
        total.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.train.grad_clip)
        self.opt.step()
        return {"loss": total.item(), "disc": l_d.item(), "gen": l_g.item(), "fm": l_fm.item(), "mel": l_mel.item(),
                **{k: v.item() for k, v in losses.items()}}

    def mel_step(self, batch: dict) -> dict:
        self.opt.zero_grad(set_to_none=True)
        with torch.autocast(self.device.type, enabled=self.scaler.is_enabled()):
            out = self.model(batch)
        self.scaler.scale(out["loss"]).backward()
        self.scaler.unscale_(self.opt)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.train.grad_clip)
        self.scaler.step(self.opt)
        self.scaler.update()
        return {k: v.item() for k, v in out.items() if torch.is_tensor(v) and v.dim() == 0}

    # -- validation ----------------------------------------------------------
    @torch.no_grad()
    def validate(self) -> dict:
        self.model.eval()
        totals: dict[str, float] = {}
        n = 0
        attn = None
        for batch in self.val_loader:
            batch = to_device(batch, self.device)
            if self.gan:
                hop, seg = self.cfg.audio.hop_length, self.cfg.model.segment_frames
                if self.cfg.model.model == "vits":
                    out = self.model(batch)
                    wav_hat, starts = out["wav_hat"], out["starts"]
                    vals = {"kl": out["kl"].item(), "duration": out["duration"].item()}
                    attn = attn if attn is not None else out["attention"][0, : int(batch["mel_lens"][0]), : int(batch["text_lens"][0])]
                else:
                    starts = [0] * batch["mel"].size(0)
                    wav_hat = self.model(batch["mel"][:, :, :seg])
                    vals = {}
                wav = torch.stack([F.pad(batch["wav"][b, s * hop:(s + seg) * hop], (0, max(0, (s + seg) * hop - batch["wav"].size(1))))
                                   for b, s in enumerate(starts)])
                vals["mel"] = F.l1_loss(self.mel_fn(wav_hat[:, : wav.size(1)]), self.mel_fn(wav)).item()
                vals["loss"] = vals["mel"]
            else:
                out = self.model(batch)
                vals = {k: v.item() for k, v in out.items() if torch.is_tensor(v) and v.dim() == 0}
                vals["mel"] = vals.get("mel_after", vals["loss"])
                if attn is None and "attention" in out:
                    attn = out["attention"][0, :, : int(batch["text_lens"][0])]
            for k, v in vals.items():
                totals[k] = totals.get(k, 0.0) + v * batch["mel"].size(0)
            n += batch["mel"].size(0)
        self.model.train()
        result = {f"val_{k}": v / max(n, 1) for k, v in totals.items()}
        if attn is not None:
            plot_alignment(attn, self.out / f"alignment_{self.step:06d}.png", f"{self.cfg.model.model} step {self.step}")
        return result

    # -- main loop -------------------------------------------------------------
    def fit(self) -> dict:
        t = self.cfg.train
        start = time.time()
        self.model.train()
        done = False
        last = {}
        while not done:
            if self.epoch >= t.epochs and not t.max_steps:
                break
            for batch in self.train_loader:
                batch = to_device(batch, self.device)
                for g in self.opt.param_groups:
                    g["lr"] = lr_at(self.step, t.lr, t.warmup_steps)
                if self.opt_d is not None:
                    for g in self.opt_d.param_groups:
                        g["lr"] = lr_at(self.step, t.lr, t.warmup_steps)
                last = self.gan_step(batch) if self.gan else self.mel_step(batch)
                self.step += 1
                if self.step % t.log_every == 0:
                    msg = " ".join(f"{k} {v:.3f}" for k, v in last.items())
                    print(f"epoch {self.epoch} step {self.step} | {msg} | {time.time() - start:.0f}s", flush=True)
                    self.curves.append({"step": self.step, "epoch": self.epoch, **last})
                over_budget = t.time_budget_hours and (time.time() - start) / 3600 > t.time_budget_hours
                if (t.max_steps and self.step >= t.max_steps) or over_budget:
                    done = True
                    break
            self.epoch += 1
            if self.epoch % t.eval_every == 0 or done or self.epoch >= t.epochs:
                val = self.validate()
                print(f"epoch {self.epoch} | " + " ".join(f"{k} {v:.4f}" for k, v in val.items()), flush=True)
                self.curves.append({"step": self.step, "epoch": self.epoch, **val})
                if val["val_mel"] < self.best:
                    self.best = val["val_mel"]
                    self.save("best.pt")
                self.save("last.pt")
                self.write_curves()
        self.save("last.pt")
        self.write_curves()
        return {"step": self.step, "epoch": self.epoch, "best_val_mel": self.best, "seconds": time.time() - start,
                "last": last}

    def write_curves(self) -> None:
        keys = sorted({k for row in self.curves for k in row})
        with open(self.out / "curves.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(self.curves)
        (self.out / "curves.json").write_text(json.dumps(self.curves))


def main(argv: list[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Train a text-to-speech model or a HiFi-GAN vocoder.")
    p.add_argument("--config", default=None, help="config.json to start from (flags override it)")
    add_config_args(p)
    a = p.parse_args(argv)
    cfg = config_from_args(a, Config.load(a.config) if a.config else None)
    summary = Trainer(cfg).fit()
    print(json.dumps({k: v for k, v in summary.items() if k != "last"}))

### 8.3 Synthesizer (`src/tts/synthesize.py`)

Text -> waveform with any acoustic model and any vocoder.

```
tts-synthesize --checkpoint checkpoints/run/best.pt --text "hello world" --out synthesized/
tts-synthesize --checkpoint ... --text-file sentences.txt --vocoder hifigan --vocoder-checkpoint checkpoints/hifigan/best.pt
```

Mel models (Tacotron 2, Transformer-TTS, FastSpeech 2) produce a log-mel that the vocoder
turns into audio: Griffin-Lim by default, or a trained HiFi-GAN. VITS produces the
waveform directly and ignores the vocoder flags.

In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch



class Synthesizer:
    def __init__(self, checkpoint: str | Path, vocoder: str = "griffinlim", vocoder_checkpoint: str | Path | None = None,
                 device: str = "cpu", griffin_lim_iters: int = 32):
        self.model, self.cfg, self.tok, state = load_checkpoint(checkpoint, device)
        self.speaker_ids = state.get("speaker_ids", {})
        self.device = device
        self.vocoder_name = vocoder
        self.gl_iters = griffin_lim_iters
        self.vocoder = None
        if vocoder == "hifigan" and self.cfg.model.model != "vits":
            if vocoder_checkpoint is None:
                raise ValueError("--vocoder hifigan needs --vocoder-checkpoint")
            self.vocoder, vcfg, _, _ = load_checkpoint(vocoder_checkpoint, device)
            assert vcfg.audio.n_mels == self.cfg.audio.n_mels, "vocoder and acoustic model must share the mel setup"

    @property
    def sample_rate(self) -> int:
        return self.cfg.audio.sample_rate

    def speaker_tensor(self, speaker: str | int | None) -> torch.Tensor | None:
        if speaker is None or self.cfg.model.n_speakers <= 1:
            return None
        idx = self.speaker_ids.get(str(speaker), speaker if isinstance(speaker, int) else 0)
        return torch.tensor([int(idx)], device=self.device)

    def vocode(self, mel: torch.Tensor) -> torch.Tensor:
        """log-mel (n_mels, T) -> waveform."""
        if self.vocoder is not None:
            with torch.no_grad():
                return self.vocoder(mel[None])[0]
        return griffin_lim(mel, self.cfg.audio, n_iter=self.gl_iters)

    @torch.no_grad()
    def mel(self, text: str, speaker=None, **kw) -> tuple[torch.Tensor, torch.Tensor]:
        """-> log-mel (n_mels, T) and alignment (T or T/r, N). Not for VITS."""
        ids = torch.tensor([self.tok.encode(text)], device=self.device)
        mel, attn = self.model.infer(ids, self.speaker_tensor(speaker), **kw)
        return mel[0], attn

    @torch.no_grad()
    def __call__(self, text: str, speaker=None, **kw) -> tuple[torch.Tensor, torch.Tensor]:
        """text -> (waveform, alignment)."""
        ids = torch.tensor([self.tok.encode(text)], device=self.device)
        if self.cfg.model.model == "vits":
            wav, attn = self.model.infer(ids, self.speaker_tensor(speaker), **kw)
            return wav, attn
        mel, attn = self.model.infer(ids, self.speaker_tensor(speaker), **kw)
        return self.vocode(mel[0]), attn


def main(argv: list[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Synthesize speech from text.")
    p.add_argument("--checkpoint", required=True)
    p.add_argument("--text", action="append", default=[], help="sentence (repeatable)")
    p.add_argument("--text-file", default=None, help="one sentence per line")
    p.add_argument("--vocoder", default="griffinlim", choices=["griffinlim", "hifigan"])
    p.add_argument("--vocoder-checkpoint", default=None)
    p.add_argument("--speaker", default=None)
    p.add_argument("--out", default="synthesized")
    p.add_argument("--device", default="cpu")
    p.add_argument("--duration-scale", type=float, default=1.0, help="fastspeech2 / vits: > 1 speaks slower")
    a = p.parse_args(argv)
    texts = list(a.text)
    if a.text_file:
        texts += [line.strip() for line in Path(a.text_file).read_text().splitlines() if line.strip()]
    if not texts:
        p.error("give --text or --text-file")
    syn = Synthesizer(a.checkpoint, a.vocoder, a.vocoder_checkpoint, a.device)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    kw = {"duration_scale": a.duration_scale} if syn.cfg.model.model in ("fastspeech2", "vits") else {}
    manifest = []
    for i, text in enumerate(texts):
        wav, _ = syn(text, a.speaker, **kw)
        path = out / f"{i:03d}.wav"
        save_wav(str(path), wav, syn.sample_rate)
        manifest.append({"audio": str(path), "text": text})
        print(f"{path}: {len(wav) / syn.sample_rate:.2f} s | {text}")
    (out / "manifest.jsonl").write_text("\n".join(json.dumps(m) for m in manifest) + "\n")

### 8.4 Metrics (`src/tts/metrics.py`)

Objective evaluation of synthesized speech, and `tts-eval`.

Numbers that say how close a synthesis is to the recording of the same sentence:

- **MCD** (mel-cepstral distortion, dB): mel-cepstra (DCT of the log-mel, coefficients
  1..12) of the two clips are aligned with **DTW** (the lengths differ), then
  `MCD = 10 / ln 10 * sqrt(2 * sum_k (c_k - c'_k)^2)` averaged along the path. Lower is
  better; good systems on LJSpeech reach ~ 5 dB, and ~ 8+ dB sounds clearly different.
- **log-F0 RMSE** (cents / semitones) over frames voiced in both clips, and the **voicing
  error** (fraction of frames where one is voiced and the other not), on the DTW path.
- **duration error**: |T_syn - T_ref| / T_ref.
- **teacher-forced mel L1**: the training loss on validation clips (needs no vocoder).
- **attention diagonality**: share of attention mass within 10% of the diagonal (AR
  models); 1.0 = perfectly monotonic, ~0.2 = not aligned.
- **WER** through an ASR model (Whisper, `pip install "tts-workbench[metrics]"`): does a
  listener understand the words? Compares the ASR transcript of the synthesis with the
  text (also reported for the recordings, as a ceiling).
- **predicted MOS** with UTMOS22 via `torch.hub` (`utmos`), a model that predicts the
  1..5 naturalness score of a listening test. Real MOS still needs humans (docs/evaluation.md).

In [ ]:
from __future__ import annotations

import argparse
import json
import math
from pathlib import Path

import numpy as np
import torch



def mel_cepstrum(log_mel: torch.Tensor, n_coef: int = 13) -> np.ndarray:
    """(n_mels, T) log-mel -> (T, n_coef) cepstra via DCT-II (c0 = energy, dropped by MCD)."""
    x = log_mel.detach().cpu().numpy().T  # (T, n_mels)
    n = x.shape[1]
    k = np.arange(n_coef)[:, None]
    basis = np.cos(math.pi * k * (np.arange(n)[None, :] + 0.5) / n) * math.sqrt(2.0 / n)
    return x @ basis.T


def dtw_path(a: np.ndarray, b: np.ndarray) -> list[tuple[int, int]]:
    """Dynamic time warping on Euclidean frame distances; returns the list of (i, j) pairs."""
    d = np.sqrt(((a[:, None, :] - b[None, :, :]) ** 2).sum(-1))
    n, m = d.shape
    acc = np.full((n + 1, m + 1), np.inf)
    acc[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            acc[i, j] = d[i - 1, j - 1] + min(acc[i - 1, j], acc[i, j - 1], acc[i - 1, j - 1])
    path, i, j = [], n, m
    while i > 0 and j > 0:
        path.append((i - 1, j - 1))
        k = int(np.argmin([acc[i - 1, j - 1], acc[i - 1, j], acc[i, j - 1]]))
        i, j = (i - 1, j - 1) if k == 0 else (i - 1, j) if k == 1 else (i, j - 1)
    return path[::-1]


def mcd(mel_syn: torch.Tensor, mel_ref: torch.Tensor) -> tuple[float, list[tuple[int, int]]]:
    """Mel-cepstral distortion in dB between two log-mels (n_mels, T), DTW-aligned. Also returns the path."""
    cs, cr = mel_cepstrum(mel_syn)[:, 1:], mel_cepstrum(mel_ref)[:, 1:]
    path = dtw_path(cs, cr)
    diff = np.array([cs[i] - cr[j] for i, j in path])
    return float((10.0 / math.log(10)) * np.mean(np.sqrt(2.0 * (diff**2).sum(-1)))), path


def f0_metrics(f0_syn: np.ndarray, f0_ref: np.ndarray, path: list[tuple[int, int]]) -> dict:
    """log-F0 RMSE in semitones over co-voiced frames + voicing decision error, along a DTW path."""
    s = np.array([f0_syn[min(i, len(f0_syn) - 1)] for i, _ in path])
    r = np.array([f0_ref[min(j, len(f0_ref) - 1)] for _, j in path])
    both = (s > 0) & (r > 0)
    vde = float(((s > 0) != (r > 0)).mean())
    if both.sum() == 0:
        return {"f0_rmse_semitones": float("nan"), "voicing_error": vde}
    semitones = 12 * np.log2(s[both] / r[both])
    return {"f0_rmse_semitones": float(np.sqrt((semitones**2).mean())), "voicing_error": vde}


def attention_diagonality(attn: torch.Tensor, band: float = 0.1) -> float:
    """attn (T, N) -> share of attention mass within `band` of the diagonal."""
    T, N = attn.shape
    t = torch.arange(T, device=attn.device)[:, None].float() / max(T - 1, 1)
    n = torch.arange(N, device=attn.device)[None, :].float() / max(N - 1, 1)
    near = ((t - n).abs() <= band).float()
    return float((attn * near).sum() / attn.sum().clamp_min(1e-8))


def word_error_rate(hyp: str, ref: str) -> float:
    h, r = normalize(hyp).replace(".", "").replace(",", "").split(), normalize(ref).replace(".", "").replace(",", "").split()
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0], d[0, :] = np.arange(len(r) + 1), np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (r[i - 1] != h[j - 1]))
    return float(d[-1, -1] / max(len(r), 1))


class WhisperASR:
    """Lazy Whisper (transformers) for intelligibility; downloads ~ 300 MB the first time."""

    def __init__(self, name: str = "openai/whisper-base", device: str = "cpu"):
        from transformers import pipeline

        self.pipe = pipeline("automatic-speech-recognition", model=name, device=device)

    def __call__(self, wav: np.ndarray, sample_rate: int) -> str:
        return self.pipe({"raw": wav.astype(np.float32), "sampling_rate": sample_rate})["text"]


class UTMOS:
    """Predicted MOS (1..5) with UTMOS22-strong via torch.hub (needs internet once)."""

    def __init__(self, device: str = "cpu"):
        self.model = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(device)
        self.device = device

    @torch.no_grad()
    def __call__(self, wav: np.ndarray, sample_rate: int) -> float:
        return float(self.model(torch.from_numpy(wav.astype(np.float32))[None].to(self.device), sample_rate)[0])


def evaluate(checkpoint: str, data_dir: str, vocoder: str = "griffinlim", vocoder_checkpoint: str | None = None,
             n: int = 10, out_dir: str | None = None, whisper: bool = False, utmos: bool = False,
             device: str = "cpu") -> dict:
    """Synthesize `n` validation sentences and compare them with the recordings."""
    syn = Synthesizer(checkpoint, vocoder, vocoder_checkpoint, device)
    cfg = syn.cfg
    mel_fn = MelSpectrogram(cfg.audio)
    _, val_items = split_items(read_manifest(data_dir), seed=cfg.train.seed)
    asr = WhisperASR(device=device) if whisper else None
    mos = UTMOS(device) if utmos else None
    rows = []
    out = Path(out_dir) if out_dir else None
    if out:
        out.mkdir(parents=True, exist_ok=True)

    for item in val_items[:n]:
        ref = load_wav(str(item.audio), cfg.audio.sample_rate)
        wav, attn = syn(item.text, item.speaker if cfg.model.n_speakers > 1 else None)
        wav_np = wav.cpu().numpy()
        mel_syn, mel_ref = mel_fn(wav[None].cpu())[0], mel_fn(torch.from_numpy(ref)[None])[0]
        score, path = mcd(mel_syn, mel_ref)
        row = {"id": item.id, "text": item.text, "mcd_db": score,
               "duration_error": abs(len(wav_np) - len(ref)) / len(ref),
               **f0_metrics(estimate_f0(wav_np, cfg.audio), estimate_f0(ref, cfg.audio), path)}
        if attn is not None and attn.dim() == 2:
            row["attention_diagonality"] = attention_diagonality(attn)
        if asr:
            row["wer"] = word_error_rate(asr(wav_np, cfg.audio.sample_rate), item.text)
            row["wer_recording"] = word_error_rate(asr(ref, cfg.audio.sample_rate), item.text)
        if mos:
            row["utmos"] = mos(wav_np, cfg.audio.sample_rate)
            row["utmos_recording"] = mos(ref, cfg.audio.sample_rate)
        rows.append(row)
        if out:
            save_wav(str(out / f"{item.id}_syn.wav"), wav_np, cfg.audio.sample_rate)
    keys = [k for k in rows[0] if isinstance(rows[0][k], float)]
    summary = {k: float(np.nanmean([r[k] for r in rows])) for k in keys}
    summary["n"] = len(rows)
    summary["model"] = cfg.model.model
    summary["vocoder"] = "none (end-to-end)" if cfg.model.model == "vits" else vocoder
    result = {"summary": summary, "items": rows}
    if out:
        (out / "metrics.json").write_text(json.dumps(result, indent=2))
    return result


def export_teacher_durations(checkpoint: str, data_dir: str, device: str = "cpu") -> int:
    """Teacher-forced Tacotron 2 attention -> integer durations per text symbol, saved to <data_dir>/durations/."""

    syn = Synthesizer(checkpoint, device=device)
    cfg = syn.cfg
    items = read_manifest(data_dir)
    ds = TTSDataset(items, cfg.audio, syn.tok, data_dir)
    out = Path(data_dir) / "durations"
    out.mkdir(exist_ok=True)
    r = cfg.model.reduction_factor
    with torch.no_grad():
        for i in range(len(ds)):
            batch = to_device(collate([ds[i]]), device)
            attn = syn.model(batch)["attention"][0]  # (steps, N)
            attn = torch.repeat_interleave(attn, r, dim=0)[: int(batch["mel_lens"][0])]
            dur = durations_from_attention(attn, int(batch["text_lens"][0]), int(batch["mel_lens"][0]))
            torch.save(dur, out / f"{ds.items[i].id}.pt")
    return len(ds)


def main(argv: list[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Evaluate a checkpoint on validation sentences.")
    p.add_argument("--checkpoint", required=True)
    p.add_argument("--data-dir", required=True)
    p.add_argument("--vocoder", default="griffinlim", choices=["griffinlim", "hifigan"])
    p.add_argument("--vocoder-checkpoint", default=None)
    p.add_argument("--n", type=int, default=10)
    p.add_argument("--out", default=None, help="folder for synthesized wavs + metrics.json")
    p.add_argument("--whisper", action="store_true", help="WER through Whisper (transformers)")
    p.add_argument("--utmos", action="store_true", help="predicted MOS through UTMOS22 (torch.hub)")
    p.add_argument("--device", default="cpu")
    p.add_argument("--export-durations", action="store_true",
                   help="instead of evaluating: write Tacotron 2 teacher durations for fastspeech2 --duration-source teacher")
    a = p.parse_args(argv)
    if a.export_durations:
        print(f"wrote durations for {export_teacher_durations(a.checkpoint, a.data_dir, a.device)} clips")
        return
    result = evaluate(a.checkpoint, a.data_dir, a.vocoder, a.vocoder_checkpoint, a.n, a.out, a.whisper, a.utmos,
                      a.device)
    print(json.dumps(result["summary"], indent=2))

## 9. Train

The config below is the repo default (a few million parameters at most) scaled by the flags of
section 2.2. Curves and alignment plots land in `OUTPUT_DIR`; the cell after the training draws them.

In [ ]:
import dataclasses
import json

cfg = Config()
for k, v in AUDIO.items():
    setattr(cfg.audio, k, v)
cfg.model.model = MODEL
cfg.model.upsample_rates = UPSAMPLE_RATES
cfg.train.data_dir = str(DATA_DIR)
cfg.train.output_dir = str(OUTPUT_DIR)
cfg.train.epochs = 8 if SMOKE_TEST else EPOCHS
cfg.train.batch_size = 16 if SMOKE_TEST else BATCH_SIZE
cfg.train.lr = LR
cfg.train.resume = RESUME
cfg.train.time_budget_hours = 0.2 if SMOKE_TEST else TIME_BUDGET_HOURS
cfg.train.amp = torch.cuda.is_available() and MODEL not in ("hifigan", "vits")
cfg.train.eval_every = 1 if SMOKE_TEST else 5
cfg.train.log_every = 10
if SMOKE_TEST:
    cfg.model.hidden = 64
    cfg.model.upsample_initial = 64
print(json.dumps(dataclasses.asdict(cfg.model), indent=1))

trainer = Trainer(cfg)
summary = trainer.fit()
print(summary)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

curves = pd.read_csv(OUTPUT_DIR / "curves.csv")
train_rows = curves[curves["loss"].notna()] if "loss" in curves else curves
val_cols = [c for c in curves.columns if c.startswith("val_")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_rows.plot(x="step", y=[c for c in ("loss", "mel", "mel_after", "kl", "duration") if c in train_rows], ax=axes[0], title="training")
curves[curves[val_cols[0]].notna()].plot(x="step", y=val_cols[:4], ax=axes[1], title="validation")
plt.show()
plots = sorted(OUTPUT_DIR.glob("alignment_*.png"))
if plots:
    from IPython.display import Image, display

    display(Image(filename=str(plots[-1])))

## 10. Results

Synthesize a few sentences, listen, and score the validation set: **MCD** (lower is better),
**log-F0 RMSE**, **duration error**, **attention diagonality** and, with Whisper and UTMOS, the
**WER** (can the words be understood?) and a **predicted MOS** (1..5, how natural does it sound?).
UTMOS is downloaded through `torch.hub` (needs internet in the session); for a human MOS see the
repo's `docs/evaluation.md`.

In [ ]:
from IPython.display import Audio, display

ckpt = OUTPUT_DIR / "best.pt"
if MODEL == "hifigan":
    print("hifigan is a vocoder: copy-synthesis of a validation clip")
    syn_v = Synthesizer(ckpt)
    item = items[0]
    ref = load_wav(str(item.audio), cfg.audio.sample_rate)
    mel = MelSpectrogram(cfg.audio)(torch.from_numpy(ref)[None])[0]
    wav = syn_v.model(mel[None])[0]
    display(Audio(ref, rate=cfg.audio.sample_rate))
    display(Audio(wav.numpy(), rate=cfg.audio.sample_rate))
else:
    syn = Synthesizer(ckpt, VOCODER, HIFIGAN_CHECKPOINT, "cuda" if torch.cuda.is_available() else "cpu")
    sentences = [it.text for it in items[:3]] if SMOKE_TEST else [
        "The quick brown fox jumps over the lazy dog.",
        "Text to speech turns written words into sound.",
        items[0].text,
    ]
    for text in sentences:
        wav, attn = syn(text)
        print(text)
        display(Audio(wav.cpu().numpy(), rate=cfg.audio.sample_rate))
    if attn is not None:
        plt.imshow(attn.cpu().numpy().T, aspect="auto", origin="lower")
        plt.xlabel("frames")
        plt.ylabel("symbols")
        plt.title("alignment of the last sentence")
        plt.show()

In [ ]:
USE_WHISPER = not SMOKE_TEST   # the toy language has no words Whisper could recognise
USE_UTMOS = not SMOKE_TEST
if MODEL != "hifigan":
    result = evaluate(str(ckpt), str(DATA_DIR), VOCODER, HIFIGAN_CHECKPOINT, n=10, out_dir=str(OUTPUT_DIR / "eval"),
                      whisper=USE_WHISPER, utmos=USE_UTMOS, device="cuda" if torch.cuda.is_available() else "cpu")
    print(json.dumps(result["summary"], indent=2))
    pd.DataFrame(result["items"]).drop(columns=["text"]).head(10)

## 11. Download the trained model

`best.pt` holds the config, tokenizer kind, speaker map and weights, so it is all `tts-synthesize`
needs: `tts-synthesize --checkpoint best.pt --text "..."`. On Kaggle, click *Save Version* to keep
`/kaggle/working`; on Colab, the cell below downloads the file.

In [ ]:
if ON_COLAB:
    from google.colab import files

    files.download(str(OUTPUT_DIR / "best.pt"))
else:
    print("checkpoint:", OUTPUT_DIR / "best.pt")